# 13. Hierarchical Chunking & Retrieval

사람이 검증한 10개 카드의 gold raw TXT를 canonical 본문으로 사용하고, gold structured JSON은 청크 경계와 metadata 보조로만 사용한다. 이 노트북은 **적재 시점**에 chunk/index와 고정 평가 질의 임베딩을 만들며, 이후 검색 셀은 저장된 임베딩과 로컬 Chroma를 사용하는 **offline retrieval simulation**이다. 실제 서비스의 검색 실행은 사용자 질의가 들어오는 사용 시점에 일어난다.

BC gold raw는 selected excerpt이고 IBK는 incomplete/ambiguous이므로 두 카드 결과를 완전 페이지 coverage로 해석하면 안 된다. 나머지 8개도 12번 감사 기준으로 시각 감사 전 `full_page_candidate`다.

경계 우선순위는 card routing overview → `[page n]` → Markdown heading → structured label-assisted benefit range다. card/page/section은 고정 overlap 0이며, benefit만 최고 일치 raw 문단의 앞뒤 한 문단을 문맥으로 포함한다. 동일 raw 범위를 선택한 structured labels는 본문 한 개로 병합하고 모든 label metadata를 canonical JSON 문자열로 보존한다.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import tempfile
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import chromadb
from chromadb.config import Settings

import chromadb
import numpy as np
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'data/ocr_benchmark/gold').is_dir())
DATA_ROOT = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
RAW_ROOT = PROJECT_ROOT / 'data/ocr_benchmark/gold/raw'
STRUCTURED_ROOT = PROJECT_ROOT / 'data/ocr_benchmark/gold/structured'
CHROMA_ROOT = DATA_ROOT / 'chroma'
EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_BATCH_SIZE = 64
MAX_EMBEDDING_ITEMS = 1000
MAX_EMBEDDING_TOKENS = 200_000
CARD_MAX_CHARS = 2_000
PAGE_MAX_CHARS = 6_000
SECTION_MAX_CHARS = 4_000
BENEFIT_MAX_CHARS = 3_000
LIVE_EMBEDDING_API = os.getenv('LIVE_EMBEDDING_API', 'false').lower() in {'1', 'true', 'yes'}
load_dotenv(PROJECT_ROOT / '.env')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
CHROMA_ROOT.mkdir(parents=True, exist_ok=True)
print({'environment': 'skn25 required', 'embedding_model': EMBEDDING_MODEL, 'live_api_guard': LIVE_EMBEDDING_API})

{'environment': 'skn25 required', 'embedding_model': 'text-embedding-3-small', 'live_api_guard': True}


In [2]:
def canonical_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))


def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def sha256_file(path):
    return sha256_bytes(path.read_bytes())


def atomic_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        json.dump(value, handle, ensure_ascii=False, indent=2)
        handle.write('\n')
    os.replace(temporary, path)


def atomic_csv(path, rows):
    rows = list(rows)
    columns = list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name)
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temporary, path)


raw_paths = sorted(RAW_ROOT.glob('*/*.txt'))
structured_paths = sorted(STRUCTURED_ROOT.glob('*/*.json'))
raw_by_key = {(path.parent.name, path.stem): path for path in raw_paths}
structured_by_key = {(path.parent.name, path.stem): path for path in structured_paths}
assert len(raw_by_key) == len(structured_by_key) == 10
assert raw_by_key.keys() == structured_by_key.keys()

def coverage_status(key):
    if key == ('BC', 'BC_Biz_AirMoney'):
        return 'selected_excerpt'
    if key == ('ibk', 'IBK_Point3.8(Credit)'):
        return 'incomplete_or_ambiguous'
    return 'full_page_candidate_unapproved'


inputs = []
for key in sorted(raw_by_key):
    structured = json.loads(structured_by_key[key].read_text(encoding='utf-8'))
    assert (structured['issuer'], structured['card_name']) == key
    inputs.append({
        'issuer': key[0], 'card_name': key[1], 'coverage_status': coverage_status(key),
        'raw_path': raw_by_key[key].relative_to(PROJECT_ROOT).as_posix(),
        'raw_sha256': sha256_file(raw_by_key[key]),
        'structured_path': structured_by_key[key].relative_to(PROJECT_ROOT).as_posix(),
        'structured_sha256': sha256_file(structured_by_key[key]),
        'structured_annotation_scope': structured.get('annotation_scope', ''),
    })
INPUT_FINGERPRINT = sha256_bytes(canonical_json(inputs).encode())
input_manifest = {
    'schema_version': 'hierarchical_retrieval_input_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'input_fingerprint': INPUT_FINGERPRINT,
    'retrieval_inputs': 'gold raw TXT + gold structured JSON only; critical rules excluded',
    'coverage_warning': 'BC selected excerpt; IBK incomplete/ambiguous; other 8 are unapproved full-page candidates.',
    'files': inputs,
}
atomic_json(DATA_ROOT / 'input_manifest.json', input_manifest)
print({'cards': len(inputs), 'raw': len(raw_paths), 'structured': len(structured_paths), 'fingerprint': INPUT_FINGERPRINT})

{'cards': 10, 'raw': 10, 'structured': 10, 'fingerprint': '536aa7fd898a5d2d9afb439720ccd203b836ebca4549bcf8af6cdce46a135875'}


In [3]:
PAGE_MARKER = re.compile(r'(?im)^\[page\s*(\d+)\]\s*$')
HEADING = re.compile(r'(?m)^\s{0,3}#{1,6}\s+(.+?)\s*$')

def normalized(value):
    return ' '.join(unicodedata.normalize('NFKC', str(value)).lower().split())


def parse_pages(text):
    matches = list(PAGE_MARKER.finditer(text))
    if not matches:
        return {1: text.strip()}
    return {
        int(match.group(1)): text[match.end():(matches[index + 1].start() if index + 1 < len(matches) else len(text))].strip()
        for index, match in enumerate(matches)
    }


def split_sections(text):
    matches = list(HEADING.finditer(text))
    if not matches:
        return [('page_body', text.strip())]
    sections = []
    if text[:matches[0].start()].strip():
        sections.append(('page_intro', text[:matches[0].start()].strip()))
    for index, match in enumerate(matches):
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        sections.append((match.group(1).strip(), text[match.start():end].strip()))
    return sections


def bounded_parts(text, limit):
    paragraphs = [part.strip() for part in re.split(r'\n\s*\n', text) if part.strip()]
    parts, current = [], ''
    for paragraph in paragraphs:
        if len(paragraph) > limit:
            if current:
                parts.append(current)
                current = ''
            parts.extend(paragraph[index:index + limit] for index in range(0, len(paragraph), limit))
        elif current and len(current) + len(paragraph) + 2 > limit:
            parts.append(current)
            current = paragraph
        else:
            current = f'{current}\n\n{paragraph}'.strip()
    if current:
        parts.append(current)
    return parts or [text[:limit]]


def label_needles(label):
    values = [label.get('surface_text'), *(label.get('context_terms') or []), *(label.get('headers') or [])]
    return [normalized(value) for value in values if value]


def benefit_excerpt(page_text, label):
    paragraphs = [part.strip() for part in re.split(r'\n\s*\n', page_text) if part.strip()]
    needles = label_needles(label)
    scores = [sum(needle in normalized(paragraph) for needle in needles) for paragraph in paragraphs]
    best = max(range(len(paragraphs)), key=lambda index: scores[index]) if paragraphs else 0
    selected = paragraphs[max(0, best - 1):best + 2] if paragraphs else [page_text]
    document = '\n\n'.join(selected)[:BENEFIT_MAX_CHARS]
    core = (paragraphs[best] if paragraphs else page_text)[:BENEFIT_MAX_CHARS]
    return document, core


def scalar_metadata(metadata):
    result = {}
    for key, value in metadata.items():
        if value is None:
            continue
        result[key] = canonical_json(value) if isinstance(value, (dict, list, tuple)) else value
    assert all(isinstance(value, (str, int, float, bool)) for value in result.values())
    return result


def chunk_id(*parts):
    return sha256_bytes('|'.join(map(str, parts)).encode())[:32]


chunks = []
benefit_overlap_characters = 0
structured_label_count = 0
for item in inputs:
    key = (item['issuer'], item['card_name'])
    raw_text = (PROJECT_ROOT / item['raw_path']).read_text(encoding='utf-8')
    structured = json.loads((PROJECT_ROOT / item['structured_path']).read_text(encoding='utf-8'))
    pages = parse_pages(raw_text)
    base = {
        'issuer': item['issuer'], 'card_name': item['card_name'], 'card_key': '/'.join(key),
        'source_path': item['raw_path'], 'structured_path': item['structured_path'],
        'coverage_status': item['coverage_status'], 'input_fingerprint': INPUT_FINGERPRINT,
    }
    headings = [match.group(1).strip() for match in HEADING.finditer(raw_text)]
    first_lines = [line.strip() for line in raw_text.splitlines() if line.strip() and not PAGE_MARKER.match(line)][:12]
    card_body = '\n'.join(dict.fromkeys([*first_lines, *headings]))[:CARD_MAX_CHARS]
    card_id = chunk_id(INPUT_FINGERPRINT, *key, 'card')
    chunks.append({'id': card_id, 'document': card_body, 'metadata': scalar_metadata({**base, 'level': 'card', 'parent_id': '', 'page_num': 0, 'section': 'card_overview'})})
    labels_by_page = defaultdict(list)
    for kind in ('field_labels', 'numeric_labels', 'table_labels'):
        for label in structured.get(kind, []):
            structured_label_count += 1
            labels_by_page[label.get('page_num', 1)].append((kind, label))
    for page_num, page_text in pages.items():
        page_label_ids = [label['id'] for _, label in labels_by_page[page_num]]
        for part_num, document in enumerate(bounded_parts(page_text, PAGE_MAX_CHARS), 1):
            page_id = chunk_id(INPUT_FINGERPRINT, *key, 'page', page_num, part_num)
            chunks.append({'id': page_id, 'document': document, 'metadata': scalar_metadata({**base, 'level': 'page', 'parent_id': card_id, 'page_num': page_num, 'part_num': part_num, 'section': 'page', 'label_ids': page_label_ids})})
        for section_num, (section, section_text) in enumerate(split_sections(page_text), 1):
            for part_num, document in enumerate(bounded_parts(section_text, SECTION_MAX_CHARS), 1):
                if not HEADING.sub('', document).strip():
                    continue
                section_id = chunk_id(INPUT_FINGERPRINT, *key, 'section', page_num, section_num, part_num)
                chunks.append({'id': section_id, 'document': document, 'metadata': scalar_metadata({**base, 'level': 'section', 'parent_id': card_id, 'page_num': page_num, 'part_num': part_num, 'section': section})})
        benefit_groups = {}
        for kind, label in labels_by_page[page_num]:
            document, core = benefit_excerpt(page_text, label)
            if not document.strip():
                continue
            group_key = sha256_bytes(normalized(document).encode())
            group = benefit_groups.setdefault(group_key, {'document':document, 'core':core, 'labels':[]})
            group['labels'].append({'kind':kind, **label})
        for group_key, group in benefit_groups.items():
            label_ids = [label['id'] for label in group['labels']]
            benefit_id = chunk_id(INPUT_FINGERPRINT, *key, 'benefit', page_num, group_key, *label_ids)
            benefit_overlap_characters += max(0, len(group['document']) - len(group['core']))
            chunks.append({'id': benefit_id, 'document': group['document'], 'metadata': scalar_metadata({**base, 'level':'benefit', 'parent_id':card_id, 'page_num':page_num, 'section':f"benefit_group:{label_ids[0]}", 'label_ids':label_ids, 'label_kinds':sorted({label['kind'] for label in group['labels']}), 'structured_metadata':group['labels']})})

assert len({chunk['id'] for chunk in chunks}) == len(chunks)
assert all(chunk['document'].strip() for chunk in chunks)
level_counts = Counter(chunk['metadata']['level'] for chunk in chunks)
with (DATA_ROOT / 'chunks.jsonl').open('w', encoding='utf-8') as handle:
    for chunk in chunks:
        handle.write(canonical_json(chunk) + '\n')
print({'chunks': len(chunks), 'levels': dict(level_counts)})

{'chunks': 327, 'levels': {'card': 10, 'page': 50, 'section': 158, 'benefit': 109}}


In [4]:
QUERIES = [
 {'id':'bc_name','category':'proper_noun','query':'biz Air Money 법인카드는 어느 카드사 상품인가?','expected_card':'BC/BC_Biz_AirMoney','expected_level':'card','required_terms':['biz Air Money']},
 {'id':'bc_numeric','category':'numeric_condition','query':'Air Money 국내외 가맹점 적립률은 얼마인가?','expected_card':'BC/BC_Biz_AirMoney','expected_level':'benefit','required_terms':['Air Money','0.2%']},
 {'id':'bc_semantic','category':'semantic','query':'적립한 포인트로 항공권 결제 금액을 차감하는 방법은?','expected_card':'BC/BC_Biz_AirMoney','expected_level':'section','required_terms':['항공권','차감청구방식']},
 {'id':'nh_name','category':'proper_noun','query':'나무 NH 카드는 어느 카드사 상품인가?','expected_card':'NH/NH_Namu_NH','expected_level':'card','required_terms':['나무','NH']},
 {'id':'nh_numeric','category':'numeric_condition','query':'나무증권 스마트 캐시백 1위 업종의 적립률은?','expected_card':'NH/NH_Namu_NH','expected_level':'benefit','required_terms':['스마트 캐시백','8%']},
 {'id':'nh_semantic','category':'semantic','query':'나무멤버스 월 이용료를 돌려받는 혜택은?','expected_card':'NH/NH_Namu_NH','expected_level':'section','required_terms':['나무멤버스','이용료']},
 {'id':'hana_name','category':'proper_noun','query':'하나 더 소호 카드는 어떤 상품인가?','expected_card':'hana/Hana_One_More_SOHO','expected_level':'card','required_terms':['하나 더 소호']},
 {'id':'hana_numeric','category':'numeric_condition','query':'하나 더 소호 운영경비 청구할인율은?','expected_card':'hana/Hana_One_More_SOHO','expected_level':'benefit','required_terms':['운영경비','5%']},
 {'id':'hana_semantic','category':'semantic','query':'해외 매장에서 결제할 때 받는 할인 혜택은?','expected_card':'hana/Hana_One_More_SOHO','expected_level':'section','required_terms':['해외 가맹점','2%']},
 {'id':'hyundai_name','category':'proper_noun','query':'the Orange 카드의 발급사는?','expected_card':'hyundai/Hyundai_The_Orange_20260330','expected_level':'card','required_terms':['the Orange']},
 {'id':'hyundai_numeric','category':'numeric_condition','query':'현대 the Orange 본인 카드 연회비는?','expected_card':'hyundai/Hyundai_The_Orange_20260330','expected_level':'benefit','required_terms':['연회비','200,000원']},
 {'id':'hyundai_semantic','category':'semantic','query':'온라인몰과 다이닝에서 추가로 M포인트를 받는 혜택은?','expected_card':'hyundai/Hyundai_The_Orange_20260330','expected_level':'section','required_terms':['온라인몰','10% M포인트']},
 {'id':'ibk_name','category':'proper_noun','query':'IBK포인트 3.8 신용카드는 어느 은행 상품인가?','expected_card':'ibk/IBK_Point3.8(Credit)','expected_level':'card','required_terms':['IBK포인트 3.8']},
 {'id':'ibk_numeric','category':'numeric_condition','query':'IBK 특별 포인트 해외 가맹점 적립률은?','expected_card':'ibk/IBK_Point3.8(Credit)','expected_level':'benefit','required_terms':['특별 포인트Ⅰ','5%']},
 {'id':'ibk_semantic','category':'semantic','query':'쿠팡과 G마켓 같은 온라인 쇼핑 혜택은?','expected_card':'ibk/IBK_Point3.8(Credit)','expected_level':'section','required_terms':['온라인 쇼핑','쿠팡']},
 {'id':'kb_name','category':'proper_noun','query':'KB국민 프랜드카드는 어느 카드사 상품인가?','expected_card':'kookmin/Kookmin_Friend_20210917','expected_level':'card','required_terms':['KB국민','프랜드카드']},
 {'id':'kb_numeric','category':'numeric_condition','query':'프랜드 주유카드의 리터당 할인 금액은?','expected_card':'kookmin/Kookmin_Friend_20210917','expected_level':'benefit','required_terms':['리터당','70원']},
 {'id':'kb_semantic','category':'semantic','query':'프랜드 항공카드로 아시아나 마일리지를 어떻게 적립하나?','expected_card':'kookmin/Kookmin_Friend_20210917','expected_level':'section','required_terms':['아시아나','1마일']},
 {'id':'lotte_name','category':'proper_noun','query':'LOCA LIKIT Eat은 어느 카드사 상품인가?','expected_card':'lotte/Lotte_LOCA_LIKIT_Eat','expected_level':'card','required_terms':['LOCA LIKIT Eat']},
 {'id':'lotte_numeric','category':'numeric_condition','query':'LOCA LIKIT Eat 음식점 결제일 할인율은?','expected_card':'lotte/Lotte_LOCA_LIKIT_Eat','expected_level':'benefit','required_terms':['음식점','60%']},
 {'id':'lotte_semantic','category':'semantic','query':'배달의 민족과 쿠팡이츠에서 받을 수 있는 혜택은?','expected_card':'lotte/Lotte_LOCA_LIKIT_Eat','expected_level':'section','required_terms':['배달의 민족','쿠팡이츠']},
 {'id':'samsung_name','category':'proper_noun','query':'삼성 iD ALL 카드는 어느 회사 상품인가?','expected_card':'samsung/Samsung_iD_ALL','expected_level':'card','required_terms':['삼성 iD ALL']},
 {'id':'samsung_numeric','category':'numeric_condition','query':'많이 쓰는 영역 자동 맞춤 할인율은?','expected_card':'samsung/Samsung_iD_ALL','expected_level':'benefit','required_terms':['자동 맞춤 할인','5%']},
 {'id':'samsung_semantic','category':'semantic','query':'주유 통신 아파트 관리비를 함께 할인하는 혜택은?','expected_card':'samsung/Samsung_iD_ALL','expected_level':'section','required_terms':['주유','이동통신','아파트 관리비']},
 {'id':'shinhan_name','category':'proper_noun','query':'토스 신한카드 Mr.Life는 어느 카드사 상품인가?','expected_card':'shinhan/Shinhan_Toss_Mr.Life_20251231','expected_level':'card','required_terms':['토스 신한카드 Mr.Life']},
 {'id':'shinhan_numeric','category':'numeric_condition','query':'Mr.Life 월납 공과금 할인율은?','expected_card':'shinhan/Shinhan_Toss_Mr.Life_20251231','expected_level':'benefit','required_terms':['월납요금','10%']},
 {'id':'shinhan_semantic','category':'semantic','query':'주말에 4대 주유소에서 받는 혜택은?','expected_card':'shinhan/Shinhan_Toss_Mr.Life_20251231','expected_level':'section','required_terms':['4대 주유소','60원']},
 {'id':'woori_name','category':'proper_noun','query':'EVERY MILE SKYPASS는 어느 카드사 상품인가?','expected_card':'woori/Woori_Classic_EVERY_MILE_SKYPASS','expected_level':'card','required_terms':['EVERY MILE SKYPASS']},
 {'id':'woori_numeric','category':'numeric_condition','query':'EVERY MILE SKYPASS 기본 마일리지 적립 기준은?','expected_card':'woori/Woori_Classic_EVERY_MILE_SKYPASS','expected_level':'benefit','required_terms':['1천원당','1마일리지']},
 {'id':'woori_semantic','category':'semantic','query':'전월 실적 없이 해외 결제로 마일리지를 더 받는 혜택은?','expected_card':'woori/Woori_Classic_EVERY_MILE_SKYPASS','expected_level':'section','required_terms':['전월실적 조건 없음','해외 가맹점']},
]

def relevant_ids(query):
    return {chunk['id'] for chunk in chunks if chunk['metadata']['card_key'] == query['expected_card'] and chunk['metadata']['level'] == query['expected_level'] and all(normalized(term) in normalized(chunk['document']) for term in query['required_terms'])}

query_relevance = {query['id']: relevant_ids(query) for query in QUERIES}
missing_relevance = {key: value for key, value in query_relevance.items() if not value}
assert not missing_relevance, missing_relevance
encoder = tiktoken.get_encoding('cl100k_base')
embedding_items = [(f"chunk:{chunk['id']}", chunk['document']) for chunk in chunks] + [(f"query:{query['id']}", query['query']) for query in QUERIES]
planned_tokens = sum(len(encoder.encode(text)) for _, text in embedding_items)
planned_requests = math.ceil(len(embedding_items) / EMBEDDING_BATCH_SIZE)
chunk_length_stats = {}
for level in ('card','page','section','benefit'):
    selected = [chunk for chunk in chunks if chunk['metadata']['level'] == level]
    characters = [len(chunk['document']) for chunk in selected]
    tokens = [len(encoder.encode(chunk['document'])) for chunk in selected]
    chunk_length_stats[level] = {'count':len(selected),'chars_min':min(characters),'chars_p50':int(np.median(characters)),'chars_p95':int(np.percentile(characters,95)),'chars_max':max(characters),'tokens_min':min(tokens),'tokens_p50':int(np.median(tokens)),'tokens_p95':int(np.percentile(tokens,95)),'tokens_max':max(tokens)}
normalized_groups = defaultdict(list)
for chunk in chunks:
    normalized_groups[normalized(chunk['document'])].append(chunk)
exact_groups = [group for group in normalized_groups.values() if len(group) > 1]
exact_same_level_excess = sum(sum(count - 1 for count in Counter(chunk['metadata']['level'] for chunk in group).values() if count > 1) for group in exact_groups)
def shingles(text):
    tokens = re.findall(r'[가-힣a-z0-9]+', normalized(text))
    return set(zip(tokens, tokens[1:], tokens[2:])) if len(tokens) >= 3 else {tuple(tokens)}
near_duplicate_pairs = 0
for left_index, left in enumerate(chunks):
    left_key, left_shingles = normalized(left['document']), shingles(left['document'])
    for right in chunks[left_index + 1:]:
        if left['metadata']['card_key'] != right['metadata']['card_key'] or left['metadata']['level'] != right['metadata']['level'] or left_key == normalized(right['document']):
            continue
        right_shingles = shingles(right['document'])
        similarity = len(left_shingles & right_shingles) / len(left_shingles | right_shingles) if left_shingles | right_shingles else 1.0
        near_duplicate_pairs += similarity >= 0.9
raw_characters = sum(len((PROJECT_ROOT / item['raw_path']).read_text(encoding='utf-8')) for item in inputs)
chunk_characters = sum(len(chunk['document']) for chunk in chunks)
duplicate_stats = {'exact_duplicate_groups_all_levels':len(exact_groups),'exact_duplicate_excess_chunks_all_levels':sum(len(group)-1 for group in exact_groups),'exact_duplicate_excess_chunks_same_level':exact_same_level_excess,'near_duplicate_pairs_same_card_level_jaccard_0_9':near_duplicate_pairs,'raw_characters':raw_characters,'chunk_characters':chunk_characters,'chunk_to_raw_character_multiplier':chunk_characters/raw_characters,'structured_labels':structured_label_count,'benefit_chunks_after_same_range_merge':level_counts['benefit'],'benefit_overlap_characters':benefit_overlap_characters,'benefit_overlap_share_of_all_chunks':benefit_overlap_characters/chunk_characters}
assert len(embedding_items) <= MAX_EMBEDDING_ITEMS
assert planned_tokens <= MAX_EMBEDDING_TOKENS
embedding_plan = {'chunks':len(chunks), 'queries':len(QUERIES), 'embedding_items':len(embedding_items), 'planned_tokens':planned_tokens, 'batch_size':EMBEDDING_BATCH_SIZE, 'planned_requests':planned_requests, 'max_items':MAX_EMBEDDING_ITEMS, 'max_tokens':MAX_EMBEDDING_TOKENS, 'chunk_length_stats':chunk_length_stats, 'duplicate_stats':duplicate_stats, 'chunking_policy':{'boundary_priority':['card routing overview from raw lines/headings','[page n] marker','Markdown heading','structured label-assisted benefit range'],'size_chars':{'card':CARD_MAX_CHARS,'page':PAGE_MAX_CHARS,'section':SECTION_MAX_CHARS,'benefit':BENEFIT_MAX_CHARS},'card_page_size_reason':'card is a compact routing overview, not a concatenated parent document; page preserves a larger canonical raw span','structured_join':'labels selecting the same normalized raw range are merged into one benefit chunk; all labels remain scalar JSON metadata'}, 'overlap_policy':{'card':'0; selective raw overview','page':'0 fixed overlap; paragraph-boundary split','section':'0 fixed overlap; heading-boundary split','benefit':'best matching raw paragraph plus one adjacent paragraph on each side'}}
print(embedding_plan)

{'chunks': 327, 'queries': 30, 'embedding_items': 357, 'planned_tokens': 180066, 'batch_size': 64, 'planned_requests': 6, 'max_items': 1000, 'max_tokens': 200000, 'chunk_length_stats': {'card': {'count': 10, 'chars_min': 372, 'chars_p50': 596, 'chars_p95': 1025, 'chars_max': 1143, 'tokens_min': 352, 'tokens_p50': 550, 'tokens_p95': 1040, 'tokens_max': 1134}, 'page': {'count': 50, 'chars_min': 7, 'chars_p50': 1135, 'chars_p95': 3949, 'chars_max': 4478, 'tokens_min': 2, 'tokens_p50': 1055, 'tokens_p95': 3503, 'tokens_max': 4157}, 'section': {'count': 158, 'chars_min': 7, 'chars_p50': 314, 'chars_p95': 1252, 'chars_max': 2423, 'tokens_min': 2, 'tokens_p50': 283, 'tokens_p95': 1127, 'tokens_max': 2247}, 'benefit': {'count': 109, 'chars_min': 20, 'chars_p50': 317, 'chars_p95': 1378, 'chars_max': 2264, 'tokens_min': 12, 'tokens_p50': 283, 'tokens_p95': 1290, 'tokens_max': 2126}}, 'duplicate_stats': {'exact_duplicate_groups_all_levels': 34, 'exact_duplicate_excess_chunks_all_levels': 41, 'exa

In [5]:
CACHE_ROOT = DATA_ROOT / 'embedding_cache' / EMBEDDING_MODEL
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
embedding_by_key = {}
usage_records = []
api_client = None
for batch_index in range(0, len(embedding_items), EMBEDDING_BATCH_SIZE):
    batch = embedding_items[batch_index:batch_index + EMBEDDING_BATCH_SIZE]
    keys, texts = zip(*batch)
    text_hashes = [sha256_bytes(text.encode()) for text in texts]
    batch_fingerprint = sha256_bytes(canonical_json({'model':EMBEDDING_MODEL,'hashes':text_hashes}).encode())
    cache_path = CACHE_ROOT / f'{batch_fingerprint}.npz'
    if cache_path.is_file():
        cached = np.load(cache_path, allow_pickle=False)
        vectors = cached['embeddings']
        assert cached['hashes'].tolist() == text_hashes and len(vectors) == len(batch)
        usage_records.append({'batch':batch_index // EMBEDDING_BATCH_SIZE + 1,'items':len(batch),'cached':True,'input_tokens':0,'batch_fingerprint':batch_fingerprint})
    else:
        if not LIVE_EMBEDDING_API:
            raise RuntimeError(f'embedding cache miss: set LIVE_EMBEDDING_API=true after reviewing plan ({cache_path.name})')
        if not os.getenv('OPENAI_API_KEY'):
            raise RuntimeError('OPENAI_API_KEY is missing; secret value was not read into any artifact')
        api_client = api_client or OpenAI(max_retries=0, timeout=120.0)
        try:
            response = api_client.embeddings.create(model=EMBEDDING_MODEL, input=list(texts), encoding_format='float')
        except Exception as error:
            atomic_json(DATA_ROOT / 'embedding_failure.json', {'at':datetime.now(timezone.utc).isoformat(),'batch':batch_index // EMBEDDING_BATCH_SIZE + 1,'error_type':type(error).__name__,'message':str(error)[:1000]})
            raise
        vectors = np.asarray([item.embedding for item in response.data], dtype=np.float32)
        file_descriptor, temporary_name = tempfile.mkstemp(dir=CACHE_ROOT, suffix='.npz')
        os.close(file_descriptor)
        np.savez_compressed(temporary_name, embeddings=vectors, hashes=np.asarray(text_hashes))
        os.replace(temporary_name, cache_path)
        usage_records.append({'batch':batch_index // EMBEDDING_BATCH_SIZE + 1,'items':len(batch),'cached':False,'input_tokens':response.usage.prompt_tokens,'total_tokens':response.usage.total_tokens,'batch_fingerprint':batch_fingerprint})
    embedding_by_key.update(dict(zip(keys, vectors)))

embedding_usage = {
    **embedding_plan, 'model':EMBEDDING_MODEL, 'executed_api_requests':sum(not row['cached'] for row in usage_records),
    'cache_hit_batches':sum(row['cached'] for row in usage_records),
    'api_input_tokens':sum(row['input_tokens'] for row in usage_records), 'batches':usage_records,
    'secret_persisted':False,
}
atomic_json(DATA_ROOT / 'embedding_usage.json', embedding_usage)
(DATA_ROOT / 'embedding_failure.json').unlink(missing_ok=True)
print({key:embedding_usage[key] for key in ('model','executed_api_requests','cache_hit_batches','api_input_tokens')})

{'model': 'text-embedding-3-small', 'executed_api_requests': 6, 'cache_hit_batches': 0, 'api_input_tokens': 180066}


In [6]:
COLLECTION_NAME = f"gold_hier_{INPUT_FINGERPRINT[:12]}_{EMBEDDING_MODEL.replace('-', '_')}"
chroma_client = chromadb.PersistentClient(path=str(CHROMA_ROOT))
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME, metadata={'embedding_model':EMBEDDING_MODEL,'input_fingerprint':INPUT_FINGERPRINT})
current_ids = {chunk['id'] for chunk in chunks}
existing = set(collection.get(include=[]).get('ids', []))
stale = sorted(existing - current_ids)
if stale:
    collection.delete(ids=stale)
for index in range(0, len(chunks), 100):
    batch = chunks[index:index + 100]
    collection.upsert(
        ids=[chunk['id'] for chunk in batch], documents=[chunk['document'] for chunk in batch],
        metadatas=[chunk['metadata'] for chunk in batch],
        embeddings=[embedding_by_key[f"chunk:{chunk['id']}"].tolist() for chunk in batch],
    )
assert collection.count() == len(chunks)
index_manifest = {'collection':COLLECTION_NAME,'persist_path':CHROMA_ROOT.relative_to(PROJECT_ROOT).as_posix(),'embedding_model':EMBEDDING_MODEL,'input_fingerprint':INPUT_FINGERPRINT,'chunks':len(chunks),'levels':dict(level_counts)}
atomic_json(DATA_ROOT / 'index_manifest.json', index_manifest)
print(index_manifest)

{'collection': 'gold_hier_536aa7fd898a_text_embedding_3_small', 'persist_path': 'notebooks/data/13_hierarchical_chunking_retrieval/chroma', 'embedding_model': 'text-embedding-3-small', 'input_fingerprint': '536aa7fd898a5d2d9afb439720ccd203b836ebca4549bcf8af6cdce46a135875', 'chunks': 327, 'levels': {'card': 10, 'page': 50, 'section': 158, 'benefit': 109}}


In [7]:
TOKEN = re.compile(r'[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*', re.IGNORECASE)
tokenized_documents = [TOKEN.findall(normalized(chunk['document'])) for chunk in chunks]
document_frequency = Counter(token for tokens in tokenized_documents for token in set(tokens))
average_length = sum(map(len, tokenized_documents)) / len(tokenized_documents)

def keyword_rank(query):
    query_tokens = TOKEN.findall(normalized(query))
    scores = []
    for chunk, tokens in zip(chunks, tokenized_documents):
        frequencies = Counter(tokens)
        score = 0.0
        for token in query_tokens:
            frequency = frequencies[token]
            if not frequency:
                continue
            inverse_frequency = math.log(1 + (len(chunks) - document_frequency[token] + 0.5) / (document_frequency[token] + 0.5))
            score += inverse_frequency * frequency * 2.5 / (frequency + 1.5 * (1 - 0.75 + 0.75 * len(tokens) / average_length))
        scores.append((chunk['id'], score))
    return [identifier for identifier, _ in sorted(scores, key=lambda item:(-item[1], item[0]))]


def vector_rank(query):
    result = collection.query(query_embeddings=[embedding_by_key[f"query:{query['id']}"].tolist()], n_results=min(50, len(chunks)), include=['distances'])
    return result['ids'][0]


def rrf_rank(*rankings, constant=60):
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, identifier in enumerate(ranking, 1):
            scores[identifier] += 1 / (constant + rank)
    return [identifier for identifier, _ in sorted(scores.items(), key=lambda item:(-item[1], item[0]))]


rankings = {}
for query in QUERIES:
    keyword = keyword_rank(query['query'])
    vector = vector_rank(query)
    rankings[query['id']] = {'keyword':keyword, 'vector':vector, 'hybrid':rrf_rank(keyword[:50], vector[:50])}
print({'queries':len(rankings), 'retrieval':'offline: cached query embeddings + local Chroma/BM25/RRF'})

{'queries': 30, 'retrieval': 'offline: cached query embeddings + local Chroma/BM25/RRF'}


In [8]:
chunk_by_id = {chunk['id']:chunk for chunk in chunks}

def evaluate_ranking(query, ranking):
    relevant = query_relevance[query['id']]
    expected_card = query['expected_card']
    hit3 = any(chunk_by_id[identifier]['metadata']['card_key'] == expected_card for identifier in ranking[:3])
    hits5 = [identifier in relevant for identifier in ranking[:5]]
    first = next((rank for rank, identifier in enumerate(ranking[:5], 1) if identifier in relevant), None)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits5, 1))
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    return {'hit_at_3':int(hit3), 'recall_at_5':sum(hits5) / len(relevant), 'mrr_at_5':1 / first if first else 0.0, 'ndcg_at_5':dcg / ideal if ideal else 0.0}


per_query_rows = []
for query in QUERIES:
    for method, ranking in rankings[query['id']].items():
        metrics = evaluate_ranking(query, ranking)
        per_query_rows.append({
            'query_id':query['id'],'category':query['category'],'query':query['query'],'method':method,
            'expected_card':query['expected_card'],'expected_level':query['expected_level'],
            'required_terms':canonical_json(query['required_terms']),'relevant_chunk_count':len(query_relevance[query['id']]),
            **metrics, 'top5_chunk_ids':canonical_json(ranking[:5]),
            'top5_cards':canonical_json([chunk_by_id[item]['metadata']['card_key'] for item in ranking[:5]]),
        })

summary_rows = []
for method in ('keyword','vector','hybrid'):
    selected = [row for row in per_query_rows if row['method'] == method]
    summary_rows.append({'method':method, **{metric:sum(row[metric] for row in selected) / len(selected) for metric in ('hit_at_3','recall_at_5','mrr_at_5','ndcg_at_5')}})
summary_rows

[{'method': 'keyword',
  'hit_at_3': 0.9333333333333333,
  'recall_at_5': 0.6166666666666667,
  'mrr_at_5': 0.3816666666666667,
  'ndcg_at_5': 0.42944653268762767},
 {'method': 'vector',
  'hit_at_3': 0.9666666666666667,
  'recall_at_5': 0.725,
  'mrr_at_5': 0.5111111111111112,
  'ndcg_at_5': 0.5370432222851463},
 {'method': 'hybrid',
  'hit_at_3': 0.9,
  'recall_at_5': 0.725,
  'mrr_at_5': 0.45388888888888884,
  'ndcg_at_5': 0.5036922394548896}]

In [9]:
hybrid_rows = [row for row in per_query_rows if row['method'] == 'hybrid']
representative_ids = [row['query_id'] for row in sorted(hybrid_rows, key=lambda row:(row['mrr_at_5'], row['hit_at_3']))[:3]]
representative_ids += [row['query_id'] for row in sorted(hybrid_rows, key=lambda row:(-row['mrr_at_5'], -row['hit_at_3']))[:3]]
evidence = []
for query_id in dict.fromkeys(representative_ids):
    query = next(item for item in QUERIES if item['id'] == query_id)
    for rank, identifier in enumerate(rankings[query_id]['hybrid'][:3], 1):
        chunk = chunk_by_id[identifier]
        evidence.append({'query_id':query_id,'query':query['query'],'rank':rank,'expected_card':query['expected_card'],'retrieved_card':chunk['metadata']['card_key'],'level':chunk['metadata']['level'],'page_num':chunk['metadata']['page_num'],'relevant_chunk':identifier in query_relevance[query_id],'evidence':normalized(chunk['document'])[:320]})
evidence

[{'query_id': 'ibk_semantic',
  'query': '쿠팡과 G마켓 같은 온라인 쇼핑 혜택은?',
  'rank': 1,
  'expected_card': 'ibk/IBK_Point3.8(Credit)',
  'retrieved_card': 'shinhan/Shinhan_Toss_Mr.Life_20251231',
  'level': 'benefit',
  'page_num': 2,
  'relevant_chunk': False,
  'evidence': '• 오후 9시 ~ 오전 9시까지 온라인 쇼핑, 택시, 식음료 10% 할인 | 구분 | 서비스 대상 | 서비스 제공 | | --- | --- | --- | | 온라인 쇼핑 | 옥션, g마켓, ak몰, 11번가, 티몬, 위메프, 쿠팡 | 구분 영역별 각각 • 일 1회 / 월 10회 할인 적용 • 1회 승인금액 1만원까지 할인 적용(1회 최대 1천원 할인)| | 택시 | 후불교통/비교통 카드 모두 적용 | | | 식음료 | 한식, 양식, 일식, 중식, 뷔페, 일반대중음식점, 패스트푸드, 커피전문점 업종 | | ※ 편의점, 병원/약국, 세탁소, 식음료는 신한카드 가맹점 업종'},
 {'query_id': 'ibk_semantic',
  'query': '쿠팡과 G마켓 같은 온라인 쇼핑 혜택은?',
  'rank': 2,
  'expected_card': 'ibk/IBK_Point3.8(Credit)',
  'retrieved_card': 'shinhan/Shinhan_Toss_Mr.Life_20251231',
  'level': 'benefit',
  'page_num': 2,
  'relevant_chunk': False,
  'evidence': '| 구분 | 서비스 대상 | 서비스 제공 | | --- | --- | --- | | 온라인 쇼핑 | 옥션, g마켓, ak몰, 11번가, 티몬, 위메프, 쿠팡 | 구분 영역별 각각 • 일 1회 / 월 10회 할인 적용 • 1회 승인금액 1만원까지 할

In [10]:
atomic_csv(DATA_ROOT / 'retrieval_per_query.csv', per_query_rows)
atomic_csv(DATA_ROOT / 'retrieval_summary.csv', summary_rows)
summary = {
    'schema_version':'hierarchical_retrieval_summary_v1','created_at':datetime.now(timezone.utc).isoformat(),
    'input_fingerprint':INPUT_FINGERPRINT,'collection':COLLECTION_NAME,'chroma_path':CHROMA_ROOT.relative_to(PROJECT_ROOT).as_posix(),
    'embedding':embedding_usage,'chunk_count':len(chunks),'chunk_levels':dict(level_counts),'chunk_length_stats':chunk_length_stats,'duplicate_stats':duplicate_stats,'chunking_policy':embedding_plan['chunking_policy'],'overlap_policy':embedding_plan['overlap_policy'],'query_count':len(QUERIES),
    'metrics':{row['method']:{key:value for key,value in row.items() if key != 'method'} for row in summary_rows},
    'metric_contract':{
        'Hit@3':'top 3 중 expected card의 청크가 하나 이상이면 1',
        'Recall@5':'expected card/level에서 required_terms를 모두 포함한 relevant chunks 중 top 5가 회수한 비율',
        'MRR@5':'첫 relevant chunk의 reciprocal rank, top 5 밖이면 0',
        'nDCG@5':'위 relevance 정의의 binary gain을 순위 할인해 ideal DCG로 정규화',
    },
    'limitations':[
        'BC corpus is selected excerpt, not complete page transcription.',
        'IBK corpus is incomplete or ambiguous.',
        'Other eight cards are full-page candidates pending visual audit.',
        'The fixed 30-query set is small and derived from the same canonical fixtures.',
        'Keyword tokenizer is deterministic whitespace/character-class tokenization, not Korean morphological analysis.',
        'Evaluation query embeddings were created and cached during ingestion; retrieval cells themselves are offline.',
    ],
    'representative_evidence':evidence,
}
atomic_json(DATA_ROOT / 'retrieval_summary.json', summary)
print({'summary_json':str((DATA_ROOT / 'retrieval_summary.json').relative_to(PROJECT_ROOT)),'summary_csv':str((DATA_ROOT / 'retrieval_summary.csv').relative_to(PROJECT_ROOT)),'metrics':summary['metrics']})

{'summary_json': 'notebooks/data/13_hierarchical_chunking_retrieval/retrieval_summary.json', 'summary_csv': 'notebooks/data/13_hierarchical_chunking_retrieval/retrieval_summary.csv', 'metrics': {'keyword': {'hit_at_3': 0.9333333333333333, 'recall_at_5': 0.6166666666666667, 'mrr_at_5': 0.3816666666666667, 'ndcg_at_5': 0.42944653268762767}, 'vector': {'hit_at_3': 0.9666666666666667, 'recall_at_5': 0.725, 'mrr_at_5': 0.5111111111111112, 'ndcg_at_5': 0.5370432222851463}, 'hybrid': {'hit_at_3': 0.9, 'recall_at_5': 0.725, 'mrr_at_5': 0.45388888888888884, 'ndcg_at_5': 0.5036922394548896}}}


## Offline retrieval ablation

기존 baseline JSON/CSV, embedding cache와 Chroma를 수정하지 않고 저장된 chunk/query embedding만 읽어 weighted RRF, hierarchy, two-stage, duplicate control, deterministic metadata filter를 비교한다. 검색 함수에는 query text와 corpus-derived metadata만 전달하고 expected card/level, required terms, category는 평가 함수에서만 사용한다. 30개 고정 질의로 설정을 비교하므로 결과 선택의 과적합 위험이 있다.

In [11]:
# Offline only: this cell never constructs an OpenAI client or writes baseline/cache/Chroma paths.
import csv, hashlib, json, math, os, re, shutil, sqlite3, tempfile, unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import chromadb
from chromadb.config import Settings
import numpy as np
PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'data/ocr_benchmark/gold').is_dir())
DATA_ROOT = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
CHROMA_ROOT = DATA_ROOT / 'chroma'
EMBEDDING_MODEL = 'text-embedding-3-small'
TOKEN = re.compile(r'[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*', re.IGNORECASE)
def normalized(value): return ' '.join(unicodedata.normalize('NFKC', str(value)).lower().split())
def sha256_bytes(value): return hashlib.sha256(value).hexdigest()
def canonical_json(value): return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
def atomic_json(path, value):
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name); json.dump(value, handle, ensure_ascii=False, indent=2); handle.write('\n')
    os.replace(temporary, path)
def atomic_csv(path, rows):
    rows = list(rows); columns = list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name); writer = csv.DictWriter(handle, fieldnames=columns); writer.writeheader(); writer.writerows(rows)
    os.replace(temporary, path)

ABLATION_JSON = DATA_ROOT / 'retrieval_ablation_summary.json'
ABLATION_SUMMARY_CSV = DATA_ROOT / 'retrieval_ablation_summary.csv'
ABLATION_QUERY_CSV = DATA_ROOT / 'retrieval_ablation_per_query.csv'

def file_tree_hash(root):
    digest = hashlib.sha256()
    for path in sorted(item for item in root.rglob('*') if item.is_file()):
        digest.update(path.relative_to(root).as_posix().encode())
        digest.update(hashlib.sha256(path.read_bytes()).digest())
    return digest.hexdigest()


def readonly_chroma_count():
    database = (CHROMA_ROOT / 'chroma.sqlite3').resolve()
    with sqlite3.connect(f'file:{database}?mode=ro', uri=True) as connection:
        return connection.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]


baseline_usage = json.loads((DATA_ROOT / 'embedding_usage.json').read_text(encoding='utf-8'))
baseline_summary = json.loads((DATA_ROOT / 'retrieval_summary.json').read_text(encoding='utf-8'))
state_before = {
    'api_requests': baseline_usage['executed_api_requests'],
    'api_input_tokens': baseline_usage['api_input_tokens'],
    'embedding_cache_hash': file_tree_hash(DATA_ROOT / 'embedding_cache'),
    'chroma_hash': file_tree_hash(CHROMA_ROOT),
    'chroma_count': readonly_chroma_count(),
}
ablation_chunks = [json.loads(line) for line in (DATA_ROOT / 'chunks.jsonl').read_text(encoding='utf-8').splitlines()]
ablation_chunk_by_id = {chunk['id']: chunk for chunk in ablation_chunks}

baseline_query_rows = list(csv.DictReader((DATA_ROOT / 'retrieval_per_query.csv').open(encoding='utf-8')))
ablation_queries = []
for row in baseline_query_rows:
    if row['method'] != 'keyword':
        continue
    ablation_queries.append({
        'id': row['query_id'], 'query': row['query'], 'category': row['category'],
        'expected_card': row['expected_card'], 'expected_level': row['expected_level'],
        'required_terms': json.loads(row['required_terms']),
    })
assert len(ablation_queries) == 30

cached_embedding_items = [(f"chunk:{chunk['id']}", chunk['document']) for chunk in ablation_chunks] + [(f"query:{query['id']}", query['query']) for query in ablation_queries]
cached_vectors_by_key = {}
for batch_start in range(0, len(cached_embedding_items), 64):
    batch = cached_embedding_items[batch_start:batch_start + 64]
    text_hashes = [sha256_bytes(text.encode()) for _, text in batch]
    batch_fingerprint = sha256_bytes(canonical_json({'model': EMBEDDING_MODEL, 'hashes': text_hashes}).encode())
    cached = np.load(DATA_ROOT / 'embedding_cache' / EMBEDDING_MODEL / f'{batch_fingerprint}.npz', allow_pickle=False)
    assert cached['hashes'].tolist() == text_hashes and len(cached['embeddings']) == len(batch)
    cached_vectors_by_key.update({key: vector for (key, _), vector in zip(batch, cached['embeddings'])})
query_vectors = {query['id']: cached_vectors_by_key[f"query:{query['id']}"] for query in ablation_queries}
temporary_chroma = tempfile.TemporaryDirectory()
temporary_chroma_root = Path(temporary_chroma.name) / 'chroma'
shutil.copytree(CHROMA_ROOT, temporary_chroma_root)
offline_chroma_client = chromadb.PersistentClient(path=str(temporary_chroma_root), settings=Settings(anonymized_telemetry=False))
offline_collection = offline_chroma_client.get_collection(json.loads((DATA_ROOT / 'index_manifest.json').read_text(encoding='utf-8'))['collection'])

ablation_tokenized = [TOKEN.findall(normalized(chunk['document'])) for chunk in ablation_chunks]
ablation_df = Counter(token for tokens in ablation_tokenized for token in set(tokens))
ablation_average_length = sum(map(len, ablation_tokenized)) / len(ablation_tokenized)

def offline_keyword_rank(query_text):
    query_tokens = TOKEN.findall(normalized(query_text))
    scores = []
    for chunk, tokens in zip(ablation_chunks, ablation_tokenized):
        frequencies, score = Counter(tokens), 0.0
        for token in query_tokens:
            frequency = frequencies[token]
            if frequency:
                inverse_frequency = math.log(1 + (len(ablation_chunks) - ablation_df[token] + 0.5) / (ablation_df[token] + 0.5))
                score += inverse_frequency * frequency * 2.5 / (frequency + 1.5 * (0.25 + 0.75 * len(tokens) / ablation_average_length))
        scores.append((chunk['id'], score))
    return [identifier for identifier, _ in sorted(scores, key=lambda item: (-item[1], item[0]))]


def offline_vector_rank(query_text, query_vector):
    del query_text  # Signature makes the allowed search input explicit; vector comes only from the approved cache.
    result = offline_collection.query(query_embeddings=[query_vector.tolist()], n_results=50, include=['distances'])
    return result['ids'][0]


def weighted_rrf(keyword_ranking, vector_ranking, vector_weight, keyword_weight, constant=60):
    scores = defaultdict(float)
    for weight, ranking in ((vector_weight, vector_ranking[:50]), (keyword_weight, keyword_ranking[:50])):
        for rank, identifier in enumerate(ranking, 1):
            scores[identifier] += weight / (constant + rank)
    return [identifier for identifier, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0]))]


base_rankings = {}
for query in ablation_queries:
    keyword = offline_keyword_rank(query['query'])
    vector = offline_vector_rank(query['query'], query_vectors[query['id']])
    base_rankings[query['id']] = {'keyword': keyword, 'vector': vector}

def strict_relevant_ids(query, levels=None):
    allowed = set(levels) if levels else None
    return {
        chunk['id'] for chunk in ablation_chunks
        if chunk['metadata']['card_key'] == query['expected_card']
        and chunk['metadata']['level'] == query['expected_level']
        and (allowed is None or chunk['metadata']['level'] in allowed)
        and all(normalized(term) in normalized(chunk['document']) for term in query['required_terms'])
    }


def evaluate_ablation(query, ranking, levels=None):
    relevant = strict_relevant_ids(query, levels)
    card_hit = int(any(ablation_chunk_by_id[identifier]['metadata']['card_key'] == query['expected_card'] for identifier in ranking[:3]))
    if not relevant:
        return {'card_hit_at_3': card_hit, 'strict_evidence_hit_at_3': None, 'recall_at_5': None, 'mrr_at_5': None, 'ndcg_at_5': None, 'strict_metric_status': 'N/A_no_relevant_chunk_in_configuration', 'strict_relevant_chunks': 0}
    hits = [identifier in relevant for identifier in ranking[:5]]
    first = next((rank for rank, hit in enumerate(hits, 1) if hit), None)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, 1))
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    return {'card_hit_at_3': card_hit, 'strict_evidence_hit_at_3': int(any(hits[:3])), 'recall_at_5': sum(hits) / len(relevant), 'mrr_at_5': 1 / first if first else 0.0, 'ndcg_at_5': dcg / ideal, 'strict_metric_status': 'available', 'strict_relevant_chunks': len(relevant)}


def aggregate_ablation(rows):
    metrics = {}
    for key in ('card_hit_at_3', 'strict_evidence_hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5'):
        available = [row[key] for row in rows if row[key] is not None]
        metrics[key] = sum(available) / len(available) if available else None
        metrics[f'{key}_denominator'] = len(available)
    return metrics


ablation_per_query = []
ablation_summaries = []

def record_experiment(family, configuration, rankings_by_query, levels=None, query_group='all'):
    rows = []
    for query in ablation_queries:
        if query_group == 'proper_noun_or_card' and not (query['category'] == 'proper_noun' or query['expected_level'] == 'card'):
            continue
        if query_group == 'evidence' and query['expected_level'] == 'card':
            continue
        ranking = rankings_by_query[query['id']]
        metrics = evaluate_ablation(query, ranking, levels)
        row = {'family': family, 'configuration': configuration, 'query_group': query_group, 'query_id': query['id'], 'query': query['query'], 'category': query['category'], 'expected_card': query['expected_card'], 'expected_level': query['expected_level'], **metrics, 'top5_chunk_ids': canonical_json(ranking[:5]), 'top5_cards': canonical_json([ablation_chunk_by_id[item]['metadata']['card_key'] for item in ranking[:5]]), 'top5_levels': canonical_json([ablation_chunk_by_id[item]['metadata']['level'] for item in ranking[:5]])}
        rows.append(row)
        ablation_per_query.append(row)
    ablation_summaries.append({'family': family, 'configuration': configuration, 'query_group': query_group, **aggregate_ablation(rows)})


# 1) Weighted RRF, reusing top 50 and k=60.
weighted_rankings = {}
for vector_weight, keyword_weight in ((0.5, 0.5), (0.6, 0.4), (0.7, 0.3), (0.8, 0.2)):
    name = f'vector_{vector_weight:.1f}_keyword_{keyword_weight:.1f}'
    weighted_rankings[name] = {query['id']: weighted_rrf(base_rankings[query['id']]['keyword'], base_rankings[query['id']]['vector'], vector_weight, keyword_weight) for query in ablation_queries}
    record_experiment('weighted_rrf', name, weighted_rankings[name])

# Baseline comparison rows use the reconstructed rankings; assert they reproduce saved metrics.
baseline_rankings = {
    'keyword': {query['id']: base_rankings[query['id']]['keyword'] for query in ablation_queries},
    'vector': {query['id']: base_rankings[query['id']]['vector'] for query in ablation_queries},
    'hybrid_rrf_0.5_0.5': weighted_rankings['vector_0.5_keyword_0.5'],
}
for name, values in baseline_rankings.items():
    record_experiment('baseline_reconstructed', name, values)
saved_method_names = {'keyword': 'keyword', 'vector': 'vector', 'hybrid_rrf_0.5_0.5': 'hybrid'}
for row in [item for item in ablation_summaries if item['family'] == 'baseline_reconstructed']:
    saved = baseline_summary['metrics'][saved_method_names[row['configuration']]]
    assert abs(row['card_hit_at_3'] - saved['hit_at_3']) < 1e-12
    for key in ('recall_at_5', 'mrr_at_5', 'ndcg_at_5'):
        assert abs(row[key] - saved[key]) < 1e-12

# 2) Hierarchy combinations. expected_level is used only inside evaluate_ablation.
hierarchy_levels = {'card': ('card',), 'page': ('page',), 'section': ('section',), 'benefit': ('benefit',), 'section+benefit': ('section', 'benefit'), 'all': ('card', 'page', 'section', 'benefit')}
selected_weighted = weighted_rankings['vector_0.7_keyword_0.3']
for name, levels in hierarchy_levels.items():
    values = {query['id']: [identifier for identifier in selected_weighted[query['id']] if ablation_chunk_by_id[identifier]['metadata']['level'] in levels] for query in ablation_queries}
    record_experiment('hierarchy', name, values, levels)

# 3) Two-stage card candidates -> section+benefit evidence. Both channels are generated for every query.
two_stage = {}
for candidate_n in (1, 3, 5):
    card_values, evidence_values = {}, {}
    for query in ablation_queries:
        card_rank = [identifier for identifier in selected_weighted[query['id']] if ablation_chunk_by_id[identifier]['metadata']['level'] == 'card']
        candidate_cards = [ablation_chunk_by_id[identifier]['metadata']['card_key'] for identifier in card_rank[:candidate_n]]
        evidence_rank = [identifier for identifier in selected_weighted[query['id']] if ablation_chunk_by_id[identifier]['metadata']['level'] in {'section', 'benefit'} and ablation_chunk_by_id[identifier]['metadata']['card_key'] in candidate_cards]
        card_values[query['id']] = card_rank
        evidence_values[query['id']] = evidence_rank
    two_stage[candidate_n] = {'card': card_values, 'evidence': evidence_values}
    record_experiment('two_stage_card', f'top_{candidate_n}', card_values, ('card',), 'proper_noun_or_card')
    record_experiment('two_stage_evidence', f'top_{candidate_n}', evidence_values, ('section', 'benefit'), 'evidence')

# 4) Duplicate controls. parent_id is currently the card parent for section/benefit; this topology limitation is reported.
def diversify(ranking, max_per_card=None, max_per_parent=None):
    output, card_counts, parent_counts = [], Counter(), Counter()
    for identifier in ranking:
        metadata = ablation_chunk_by_id[identifier]['metadata']
        card, parent = metadata['card_key'], metadata.get('parent_id', '')
        if max_per_card is not None and card_counts[card] >= max_per_card:
            continue
        if max_per_parent is not None and parent and parent_counts[parent] >= max_per_parent:
            continue
        output.append(identifier)
        card_counts[card] += 1
        if parent:
            parent_counts[parent] += 1
    return output

duplicate_controls = {'none': (None, None), 'card_max_1': (1, None), 'card_max_2': (2, None), 'card_max_2_parent_max_1': (2, 1)}
evidence_all = {query['id']: [identifier for identifier in selected_weighted[query['id']] if ablation_chunk_by_id[identifier]['metadata']['level'] in {'section', 'benefit'}] for query in ablation_queries}
for name, (card_cap, parent_cap) in duplicate_controls.items():
    values = {query_id: diversify(ranking, card_cap, parent_cap) for query_id, ranking in evidence_all.items()}
    record_experiment('duplicate_control', name, values, ('section', 'benefit'), 'evidence')

# 5) Deterministic metadata filter: aliases are corpus metadata, not evaluation labels.
CARD_ALIASES = {
    'BC/BC_Biz_AirMoney': ('biz air money', 'air money'),
    'NH/NH_Namu_NH': ('나무 nh', '나무증권'),
    'hana/Hana_One_More_SOHO': ('하나 더 소호',),
    'hyundai/Hyundai_The_Orange_20260330': ('the orange',),
    'ibk/IBK_Point3.8(Credit)': ('ibk포인트 3.8', 'ibk 특별 포인트'),
    'kookmin/Kookmin_Friend_20210917': ('kb국민 프랜드', '프랜드 주유카드', '프랜드 항공카드'),
    'lotte/Lotte_LOCA_LIKIT_Eat': ('loca likit eat',),
    'samsung/Samsung_iD_ALL': ('삼성 id all',),
    'shinhan/Shinhan_Toss_Mr.Life_20251231': ('토스 신한카드', 'mr.life'),
    'woori/Woori_Classic_EVERY_MILE_SKYPASS': ('every mile skypass',),
}
def detected_card_from_query(query_text):
    text = normalized(query_text)
    matches = [card for card, aliases in CARD_ALIASES.items() if any(normalized(alias) in text for alias in aliases)]
    return matches[0] if len(matches) == 1 else None


def card_intent_from_query(query_text):
    text = normalized(query_text)
    return any(phrase in text for phrase in ('어느 카드사', '어느 은행', '어느 회사', '발급사', '어떤 상품'))


metadata_filtered = {}
metadata_detection = {}
for query in ablation_queries:
    detected = detected_card_from_query(query['query'])
    metadata_detection[query['id']] = detected
    metadata_filtered[query['id']] = [identifier for identifier in selected_weighted[query['id']] if detected is None or ablation_chunk_by_id[identifier]['metadata']['card_key'] == detected]
record_experiment('metadata_filter', 'unique_explicit_alias_else_unfiltered', metadata_filtered)

# 6) Predeclared final: 0.7/0.3 RRF, unique explicit alias, card intent routing, otherwise top-3 card -> evidence, card cap 2.
final_rankings = {}
for query in ablation_queries:
    detected = metadata_detection[query['id']]
    ranking = metadata_filtered[query['id']]
    card_rank = [identifier for identifier in ranking if ablation_chunk_by_id[identifier]['metadata']['level'] == 'card']
    candidate_cards = ([detected] if detected else [ablation_chunk_by_id[identifier]['metadata']['card_key'] for identifier in card_rank[:3]])
    if card_intent_from_query(query['query']):
        final_rankings[query['id']] = card_rank
    else:
        evidence_rank = [identifier for identifier in ranking if ablation_chunk_by_id[identifier]['metadata']['level'] in {'section', 'benefit'} and ablation_chunk_by_id[identifier]['metadata']['card_key'] in candidate_cards]
        final_rankings[query['id']] = diversify(evidence_rank, max_per_card=2)
record_experiment('final', 'rrf_0.7_0.3_alias_two_stage_n3_card2', final_rankings)

state_after = {
    'api_requests': json.loads((DATA_ROOT / 'embedding_usage.json').read_text(encoding='utf-8'))['executed_api_requests'],
    'api_input_tokens': json.loads((DATA_ROOT / 'embedding_usage.json').read_text(encoding='utf-8'))['api_input_tokens'],
    'embedding_cache_hash': file_tree_hash(DATA_ROOT / 'embedding_cache'),
    'chroma_hash': file_tree_hash(CHROMA_ROOT),
    'chroma_count': readonly_chroma_count(),
}
temporary_chroma.cleanup()
assert state_after == state_before

leakage_audit = {
    'status': 'passed',
    'retrieval_allowed_inputs': ['query string', 'corpus chunk text/scalar metadata', 'cached query/chunk embeddings', 'corpus-derived CARD_ALIASES'],
    'retrieval_prohibited_inputs': ['expected_card', 'expected_level', 'required_terms', 'category'],
    'implementation_contract': 'Rank generation passes only query[query] plus cached/base rankings to retrieval functions. Gold fields enter evaluate_ablation, strict_relevant_ids, record grouping, and report rows only.',
    'metadata_filter_rule': 'Apply only when query text matches aliases for exactly one card; ambiguous or absent matches remain unfiltered.',
    'intent_rule': 'Card routing uses only fixed query phrases: 어느 카드사/은행/회사, 발급사, 어떤 상품.',
}

ablation_result = {
    'schema_version': 'retrieval_ablation_v1',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'execution': {'network_calls': 0, 'openai_calls': 0, 'embedding_cache_and_chroma_unchanged': True, 'state_before': state_before, 'state_after': state_after},
    'settings': {
        'rrf': {'k': 60, 'source_rank_depth': 50, 'weights': ['0.5:0.5', '0.6:0.4', '0.7:0.3', '0.8:0.2']},
        'hierarchy': {key: list(value) for key, value in hierarchy_levels.items()},
        'two_stage_candidate_n': [1, 3, 5],
        'duplicate_controls': {key: {'max_per_card': value[0], 'max_per_parent': value[1]} for key, value in duplicate_controls.items()},
        'tie_breaking': 'RRF score descending, then chunk ID ascending; vector squared-L2 ascending then ID; keyword BM25 descending then ID.',
        'selection_criterion': 'Predeclared final uses vector-forward 0.7/0.3 for semantic recall, top-3 candidate cards for recall/diversity, unique explicit metadata alias when available, and max two evidence chunks per card.',
    },
    'metric_contract': {
        'card_hit_at_3': 'Any top-3 chunk has expected_card.',
        'strict_evidence_hit_at_3': 'Any top-3 chunk is relevant under the unchanged expected_card + expected_level + required_terms contract.',
        'N/A': 'A hierarchy configuration has no chunk at the gold strict evidence level; excluded from that metric denominator.',
    },
    'leakage_audit': leakage_audit,
    'summaries': ablation_summaries,
    'limitations': [
        'All configurations and the final selection are observed on the same 30-query set; an independent holdout is required to estimate generalization and avoid selection overfit.',
        'CARD_ALIASES are deterministic corpus metadata but manually curated; unseen spelling variants remain unfiltered.',
        'section and benefit parent_id currently point to the card, so parent_max_1 is effectively a one-result-per-card constraint rather than section-local deduplication.',
        'BC is selected excerpt, IBK is incomplete/ambiguous, and the other eight cards remain visually unaudited full-page candidates.',
    ],
}
atomic_csv(ABLATION_QUERY_CSV, ablation_per_query)
atomic_csv(ABLATION_SUMMARY_CSV, ablation_summaries)
atomic_json(ABLATION_JSON, ablation_result)
final_summary = next(row for row in ablation_summaries if row['family'] == 'final')
print({'offline': True, 'api_calls': 0, 'cache_chroma_unchanged': state_before == state_after, 'experiments': len(ablation_summaries), 'per_query_rows': len(ablation_per_query), 'final': final_summary})

{'offline': True, 'api_calls': 0, 'cache_chroma_unchanged': True, 'experiments': 25, 'per_query_rows': 620, 'final': {'family': 'final', 'configuration': 'rrf_0.7_0.3_alias_two_stage_n3_card2', 'query_group': 'all', 'card_hit_at_3': 0.9, 'card_hit_at_3_denominator': 30, 'strict_evidence_hit_at_3': 0.7, 'strict_evidence_hit_at_3_denominator': 30, 'recall_at_5': 0.625, 'recall_at_5_denominator': 30, 'mrr_at_5': 0.65, 'mrr_at_5_denominator': 30, 'ndcg_at_5': 0.6115867824726673, 'ndcg_at_5_denominator': 30}}


## True two-stage offline re-retrieval

이 실험은 기존 전체 순위를 후보 카드로 필터링하는 방식과 구분한다. Stage 1은 card chunk 10개만 cached vector로 재정렬하고, Stage 2는 Top-3 카드의 section+benefit subset 안에서 vector distance와 BM25 통계를 새로 계산한다. API와 Chroma write는 사용하지 않는다. 기존 30개 질의는 development set이며 holdout이 아니다.

In [12]:
# Standalone offline cell: reads protected assets and writes only retrieval_true_two_stage_* outputs.
import csv, hashlib, json, math, os, re, shutil, sqlite3, tempfile, unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import numpy as np

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'data/ocr_benchmark/gold').is_dir())
DATA_ROOT = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
CHROMA_ROOT = DATA_ROOT / 'chroma'
EMBEDDING_MODEL = 'text-embedding-3-small'
TOKEN = re.compile(r'[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*', re.IGNORECASE)
normalized = lambda value: ' '.join(unicodedata.normalize('NFKC', str(value)).lower().split())
sha256_bytes = lambda value: hashlib.sha256(value).hexdigest()
canonical_json = lambda value: json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))

def atomic_json(path, value):
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name); json.dump(value, handle, ensure_ascii=False, indent=2); handle.write('\n')
    os.replace(temporary, path)


def atomic_csv(path, rows):
    rows = list(rows); columns = list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name); writer = csv.DictWriter(handle, fieldnames=columns); writer.writeheader(); writer.writerows(rows)
    os.replace(temporary, path)


def tree_hash(root):
    digest = hashlib.sha256()
    for path in sorted(item for item in root.rglob('*') if item.is_file()):
        digest.update(path.relative_to(root).as_posix().encode()); digest.update(hashlib.sha256(path.read_bytes()).digest())
    return digest.hexdigest()


def chroma_count():
    database = (CHROMA_ROOT / 'chroma.sqlite3').resolve()
    with sqlite3.connect(f'file:{database}?mode=ro', uri=True) as connection:
        return connection.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]


PROTECTED_FILES = [
    'retrieval_summary.json', 'retrieval_summary.csv', 'retrieval_per_query.csv',
    'retrieval_ablation_summary.json', 'retrieval_ablation_summary.csv', 'retrieval_ablation_per_query.csv',
]
usage_before = json.loads((DATA_ROOT / 'embedding_usage.json').read_text(encoding='utf-8'))
state_before = {
    'api_requests': usage_before['executed_api_requests'], 'api_input_tokens': usage_before['api_input_tokens'],
    'embedding_cache_hash': tree_hash(DATA_ROOT / 'embedding_cache'), 'chroma_hash': tree_hash(CHROMA_ROOT),
    'chroma_count': chroma_count(),
    'protected_file_hashes': {name: hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest() for name in PROTECTED_FILES},
}
chunks = [json.loads(line) for line in (DATA_ROOT / 'chunks.jsonl').read_text(encoding='utf-8').splitlines()]
chunk_by_id = {chunk['id']: chunk for chunk in chunks}

baseline_rows = list(csv.DictReader((DATA_ROOT / 'retrieval_per_query.csv').open(encoding='utf-8')))
evaluation_by_id = {}
for row in baseline_rows:
    if row['method'] == 'keyword':
        evaluation_by_id[row['query_id']] = {'query_id': row['query_id'], 'query': row['query'], 'category': row['category'], 'expected_card': row['expected_card'], 'expected_level': row['expected_level'], 'required_terms': json.loads(row['required_terms'])}
assert len(evaluation_by_id) == 30
search_queries = {query_id: item['query'] for query_id, item in evaluation_by_id.items()}

cached_items = [(f"chunk:{chunk['id']}", chunk['document']) for chunk in chunks] + [(f"query:{query_id}", text) for query_id, text in search_queries.items()]
vectors_by_key = {}
for batch_start in range(0, len(cached_items), 64):
    batch = cached_items[batch_start:batch_start + 64]
    hashes = [sha256_bytes(text.encode()) for _, text in batch]
    fingerprint = sha256_bytes(canonical_json({'model': EMBEDDING_MODEL, 'hashes': hashes}).encode())
    cached = np.load(DATA_ROOT / 'embedding_cache' / EMBEDDING_MODEL / f'{fingerprint}.npz', allow_pickle=False)
    assert cached['hashes'].tolist() == hashes and len(cached['embeddings']) == len(batch)
    vectors_by_key.update({key: vector for (key, _), vector in zip(batch, cached['embeddings'])})
chunk_vectors = {chunk['id']: vectors_by_key[f"chunk:{chunk['id']}"] for chunk in chunks}
query_vectors = {query_id: vectors_by_key[f"query:{query_id}"] for query_id in search_queries}
card_ids = [chunk['id'] for chunk in chunks if chunk['metadata']['level'] == 'card']
evidence_ids = [chunk['id'] for chunk in chunks if chunk['metadata']['level'] in {'section', 'benefit'}]

def vector_rank_subset(query_vector, candidate_ids):
    scores = [(identifier, float(np.sum((chunk_vectors[identifier] - query_vector) ** 2))) for identifier in candidate_ids]
    return [identifier for identifier, _ in sorted(scores, key=lambda item: (item[1], item[0]))]


def bm25_rank_subset(query_text, candidate_ids):
    tokenized = {identifier: TOKEN.findall(normalized(chunk_by_id[identifier]['document'])) for identifier in candidate_ids}
    document_frequency = Counter(token for tokens in tokenized.values() for token in set(tokens))
    average_length = sum(map(len, tokenized.values())) / len(tokenized) if tokenized else 1.0
    query_tokens, scores = TOKEN.findall(normalized(query_text)), []
    for identifier in candidate_ids:
        tokens, frequencies, score = tokenized[identifier], Counter(tokenized[identifier]), 0.0
        for token in query_tokens:
            frequency = frequencies[token]
            if frequency:
                inverse_frequency = math.log(1 + (len(candidate_ids) - document_frequency[token] + 0.5) / (document_frequency[token] + 0.5))
                score += inverse_frequency * frequency * 2.5 / (frequency + 1.5 * (0.25 + 0.75 * len(tokens) / average_length))
        scores.append((identifier, score))
    return [identifier for identifier, _ in sorted(scores, key=lambda item: (-item[1], item[0]))]


def weighted_rrf(keyword_ranking, vector_ranking, vector_weight, keyword_weight, constant=60):
    scores = defaultdict(float)
    for weight, ranking in ((vector_weight, vector_ranking[:50]), (keyword_weight, keyword_ranking[:50])):
        for rank, identifier in enumerate(ranking, 1):
            scores[identifier] += weight / (constant + rank)
    return [identifier for identifier, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0]))]


# Baselines are loaded verbatim from immutable baseline top-5 rows, avoiding any change to their published result contract.
full_vector = {row['query_id']: json.loads(row['top5_chunk_ids']) for row in baseline_rows if row['method'] == 'vector'}
full_rrf_equal = {row['query_id']: json.loads(row['top5_chunk_ids']) for row in baseline_rows if row['method'] == 'hybrid'}
assert len(full_vector) == len(full_rrf_equal) == 30

# Previous final is loaded as a distinct, already-materialized filtered method; it is not a true re-retrieval result.
previous_rows = list(csv.DictReader((DATA_ROOT / 'retrieval_ablation_per_query.csv').open(encoding='utf-8')))
existing_rrf_filtered = {row['query_id']: json.loads(row['top5_chunk_ids']) for row in previous_rows if row['family'] == 'final'}
assert len(existing_rrf_filtered) == 30

stage1_rankings, stage2_candidates = {}, {}
for query_id in search_queries:
    stage1 = vector_rank_subset(query_vectors[query_id], card_ids)
    candidate_cards = {chunk_by_id[identifier]['metadata']['card_key'] for identifier in stage1[:3]}
    stage1_rankings[query_id] = stage1
    stage2_candidates[query_id] = [identifier for identifier in evidence_ids if chunk_by_id[identifier]['metadata']['card_key'] in candidate_cards]

true_v_to_v, true_v_to_keyword, true_v_to_rrf_equal, true_v_to_rrf_07 = {}, {}, {}, {}
for query_id, query_text in search_queries.items():
    candidates = stage2_candidates[query_id]
    vector = vector_rank_subset(query_vectors[query_id], candidates)
    keyword = bm25_rank_subset(query_text, candidates)
    true_v_to_v[query_id] = vector
    true_v_to_keyword[query_id] = keyword
    true_v_to_rrf_equal[query_id] = weighted_rrf(keyword, vector, 0.5, 0.5)
    true_v_to_rrf_07[query_id] = weighted_rrf(keyword, vector, 0.7, 0.3)

CONFIGURATIONS = {
    'full_vector': {'rankings': full_vector, 'uses_true_stage1': False},
    'full_rrf_equal': {'rankings': full_rrf_equal, 'uses_true_stage1': False},
    'existing_rrf_filtered': {'rankings': existing_rrf_filtered, 'uses_true_stage1': False},
    'true_v_to_v': {'rankings': true_v_to_v, 'uses_true_stage1': True},
    'true_v_to_keyword': {'rankings': true_v_to_keyword, 'uses_true_stage1': True},
    'true_v_to_rrf_equal': {'rankings': true_v_to_rrf_equal, 'uses_true_stage1': True},
    'true_v_to_rrf_0.7_0.3': {'rankings': true_v_to_rrf_07, 'uses_true_stage1': True},
}

def relevant_ids(evaluation):
    return {chunk['id'] for chunk in chunks if chunk['metadata']['card_key'] == evaluation['expected_card'] and chunk['metadata']['level'] == evaluation['expected_level'] and all(normalized(term) in normalized(chunk['document']) for term in evaluation['required_terms'])}


def retrieval_metrics(evaluation, ranking):
    relevant = relevant_ids(evaluation)
    hits = [identifier in relevant for identifier in ranking[:5]]
    first = next((rank for rank, hit in enumerate(hits, 1) if hit), None)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, 1))
    ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    return {
        'card_hit_at_3': int(any(chunk_by_id[item]['metadata']['card_key'] == evaluation['expected_card'] for item in ranking[:3])),
        'strict_evidence_hit_at_3': int(any(hits[:3])), 'recall_at_5': sum(hits) / len(relevant),
        'mrr_at_5': 1 / first if first else 0.0, 'ndcg_at_5': dcg / ideal if ideal else 0.0,
    }


per_query_rows = []
for configuration, details in CONFIGURATIONS.items():
    for query_id, evaluation in evaluation_by_id.items():
        card_question = evaluation['expected_level'] == 'card'
        output_ranking = stage1_rankings[query_id] if details['uses_true_stage1'] and card_question else details['rankings'][query_id]
        stage1 = stage1_rankings[query_id] if details['uses_true_stage1'] else []
        metrics = retrieval_metrics(evaluation, output_ranking)
        per_query_rows.append({
            'configuration': configuration, 'query_id': query_id, 'question_group': 'card_question' if card_question else 'evidence_question', 'category': evaluation['category'],
            'query': evaluation['query'], 'expected_card': evaluation['expected_card'], 'expected_level': evaluation['expected_level'],
            'stage1_used': details['uses_true_stage1'],
            'stage1_card_hit_at_1': int(chunk_by_id[stage1[0]]['metadata']['card_key'] == evaluation['expected_card']) if stage1 else None,
            'stage1_card_hit_at_3': int(any(chunk_by_id[item]['metadata']['card_key'] == evaluation['expected_card'] for item in stage1[:3])) if stage1 else None,
            'stage1_oracle_ceiling': int(any(chunk_by_id[item]['metadata']['card_key'] == evaluation['expected_card'] for item in card_ids)) if stage1 else None,
            **metrics, 'top5_chunk_ids': canonical_json(output_ranking[:5]),
            'top5_cards': canonical_json([chunk_by_id[item]['metadata']['card_key'] for item in output_ranking[:5]]),
            'top5_levels': canonical_json([chunk_by_id[item]['metadata']['level'] for item in output_ranking[:5]]),
        })

def aggregate(selected):
    result = {}
    for metric in ('stage1_card_hit_at_1', 'stage1_card_hit_at_3', 'stage1_oracle_ceiling', 'card_hit_at_3', 'strict_evidence_hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5'):
        values = [row[metric] for row in selected if row[metric] is not None]
        result[metric] = sum(values) / len(values) if values else None
        result[f'{metric}_denominator'] = len(values)
    return result


summary_rows = []
for configuration in CONFIGURATIONS:
    configuration_rows = [row for row in per_query_rows if row['configuration'] == configuration]
    groups = {
        'all': configuration_rows,
        'card_question': [row for row in configuration_rows if row['question_group'] == 'card_question'],
        'evidence_question': [row for row in configuration_rows if row['question_group'] == 'evidence_question'],
        **{f'category_{category}': [row for row in configuration_rows if row['category'] == category] for category in ('proper_noun', 'numeric_condition', 'semantic')},
    }
    for group, selected in groups.items():
        summary_rows.append({'configuration': configuration, 'question_group': group, **aggregate(selected)})

# Failure transitions compare strict top-3 against full_vector without influencing retrieval.
row_index = {(row['configuration'], row['query_id']): row for row in per_query_rows}
failure_transitions = {}
for configuration in CONFIGURATIONS:
    if configuration == 'full_vector':
        continue
    improved, worsened = [], []
    for query_id in evaluation_by_id:
        baseline_hit = row_index[('full_vector', query_id)]['strict_evidence_hit_at_3']
        current_hit = row_index[(configuration, query_id)]['strict_evidence_hit_at_3']
        if current_hit > baseline_hit: improved.append(query_id)
        if current_hit < baseline_hit: worsened.append(query_id)
    failure_transitions[configuration] = {'improved_vs_full_vector': improved, 'worsened_vs_full_vector': worsened}

usage_after = json.loads((DATA_ROOT / 'embedding_usage.json').read_text(encoding='utf-8'))
state_after = {
    'api_requests': usage_after['executed_api_requests'], 'api_input_tokens': usage_after['api_input_tokens'],
    'embedding_cache_hash': tree_hash(DATA_ROOT / 'embedding_cache'), 'chroma_hash': tree_hash(CHROMA_ROOT),
    'chroma_count': chroma_count(),
    'protected_file_hashes': {name: hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest() for name in PROTECTED_FILES},
}
assert state_after == state_before
result = {
    'schema_version': 'retrieval_true_two_stage_v1', 'created_at': datetime.now(timezone.utc).isoformat(),
    'execution': {'network_calls': 0, 'openai_calls': 0, 'state_before': state_before, 'state_after': state_after, 'protected_assets_unchanged': True},
    'configuration_contract': {
        'stage1': 'Cached vector squared-L2 over exactly 10 card chunks; Top-3 unique cards.',
        'stage2_vector': 'Recompute cached vector distance inside candidate cards section+benefit subset.',
        'stage2_keyword': 'Recompute BM25 document frequency and average length inside candidate subset.',
        'stage2_rrf': 'Fuse the freshly recomputed subset keyword/vector rankings with k=60.',
        'full_baselines': 'Published top-5 rankings loaded verbatim from immutable retrieval_per_query.csv.',
        'existing_rrf_filtered': 'Previous alias/two-stage/card-cap final top-5 loaded from immutable ablation output; not true subset re-retrieval.',
        'tie_breaking': 'Vector distance ascending then ID; BM25 descending then ID; RRF score descending then ID.',
    },
    'metric_contract': {
        'stage1_card_hit_at_1_3': 'Expected card among card-only vector top 1/3; N/A for methods without stage1.',
        'stage1_oracle_ceiling': 'Expected card has a card chunk in the 10-card stage1 corpus.',
        'all_group': 'Card questions use stage1 card ranking for true methods; evidence questions use stage2 ranking. Every metric stores its denominator.',
        'strict_relevance': 'Unchanged expected_card + expected_level + required_terms contract.',
    },
    'leakage_audit': {
        'status': 'passed',
        'search_inputs': ['query text', 'cached query/chunk vectors', 'chunk text', 'chunk level/card metadata'],
        'gold_fields_excluded_from_search': ['expected_card', 'expected_level', 'required_terms', 'category'],
        'implementation': 'search_queries is separated from evaluation_by_id; all rankings and candidate subsets are built before retrieval_metrics reads gold.',
        'alias_routing': 'Excluded from all true_* configurations; existing_rrf_filtered is reported separately.',
    },
    'development_set_warning': 'The same 30 queries are a development set, not a holdout; configuration comparisons may overfit.',
    'summaries': summary_rows, 'failure_transitions': failure_transitions,
    'limitations': ['BC selected excerpt and IBK incomplete/ambiguous coverage remain.', 'Only one card chunk exists per card, so stage1 oracle ceiling is structurally 1.0.', 'Top-3 candidate selection cannot recover evidence when the expected card is outside stage1 Top-3.'],
}
atomic_csv(DATA_ROOT / 'retrieval_true_two_stage_per_query.csv', per_query_rows)
atomic_csv(DATA_ROOT / 'retrieval_true_two_stage_summary.csv', summary_rows)
atomic_json(DATA_ROOT / 'retrieval_true_two_stage_summary.json', result)
print({'offline': True, 'api_calls': 0, 'protected_unchanged': state_before == state_after, 'configurations': len(CONFIGURATIONS), 'summary_rows': len(summary_rows), 'per_query_rows': len(per_query_rows)})

{'offline': True, 'api_calls': 0, 'protected_unchanged': True, 'configurations': 7, 'summary_rows': 42, 'per_query_rows': 210}


## Offline search normalization and metadata fields

JSON/metadata는 저장만으로 검색에 사용되지 않는다. 이 실험은 동일 30개 development 질의에서 raw BM25, 원문+질의 공통 결정론적 정규화, 안전한 비정답 metadata(`issuer`, `card_name`, `page_num`), 수기 structured oracle field와 각각의 persisted Chroma baseline vector RRF를 비교한다. Oracle 외 검색은 `structured_metadata`, `label_ids`, structured benefit section을 읽지 않는다. 다만 공통 chunk 본문과 경계 자체는 structured-assisted chunking의 영향을 받았다는 한계가 있다. Vector 순위는 현재 보호 원본의 byte-for-byte 임시 snapshot에 cached query embedding을 넣어 `collection.query`로 읽고 published top-5와 30/30 exact를 강제한다. 최초 direct `PersistentClient` 연결로 source tree hash가 `a25c…6174`에서 `f2d7…1ac8b`로 바뀐 integrity incident가 있었으며, count·collection·published top-5는 같지만 사전 top 6–50을 저장하지 않아 전체 RRF 순위의 사전 동일성은 증명할 수 없다. 현재 RRF는 incident 이후 `f2d7…1ac8b` snapshot 기반이다. Oracle은 운영 설정이 아니라 gold 값·context를 검색 field로 노출한 성능 상한이며, 영구 chunk 본문이나 Chroma에는 주입하지 않는다.

In [14]:
# Standalone offline cell. Writes only retrieval_search_normalization_* files.
import csv, hashlib, json, math, os, re, shutil, sqlite3, tempfile, unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path
import chromadb
from chromadb.config import Settings
import numpy as np

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'data/ocr_benchmark/gold').is_dir())
DATA_ROOT = PROJECT_ROOT / 'notebooks/data/13_hierarchical_chunking_retrieval'
CHROMA_ROOT = DATA_ROOT / 'chroma'
EMBEDDING_MODEL = 'text-embedding-3-small'
RAW_TOKEN = re.compile(r'[가-힣a-z0-9]+(?:[.,%+~-][가-힣a-z0-9]+)*', re.IGNORECASE)
normalized_text = lambda value: ' '.join(unicodedata.normalize('NFKC', str(value)).lower().split())
canonical_json = lambda value: json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))
sha256_bytes = lambda value: hashlib.sha256(value).hexdigest()

def atomic_json(path, value):
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name); json.dump(value, handle, ensure_ascii=False, indent=2); handle.write('\n')
    os.replace(temporary, path)


def atomic_csv(path, rows):
    rows = list(rows); columns = list(dict.fromkeys(key for row in rows for key in row))
    with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as handle:
        temporary = Path(handle.name); writer = csv.DictWriter(handle, fieldnames=columns); writer.writeheader(); writer.writerows(rows)
    os.replace(temporary, path)


def tree_hash(root):
    digest = hashlib.sha256()
    for path in sorted(item for item in root.rglob('*') if item.is_file()):
        digest.update(path.relative_to(root).as_posix().encode()); digest.update(hashlib.sha256(path.read_bytes()).digest())
    return digest.hexdigest()


def chroma_count():
    database = (CHROMA_ROOT / 'chroma.sqlite3').resolve()
    with sqlite3.connect(f'file:{database}?mode=ro', uri=True) as connection:
        return connection.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]


OUTPUT_PREFIX = 'retrieval_search_normalization_'
def load_prior_regression(per_query_path, summary_path):
    if not (per_query_path.is_file() and summary_path.is_file()): return None, None
    return list(csv.DictReader(per_query_path.open(encoding='utf-8'))), json.loads(summary_path.read_text(encoding='utf-8'))


prior_per_query_path = DATA_ROOT / 'retrieval_search_normalization_per_query.csv'
prior_summary_path = DATA_ROOT / 'retrieval_search_normalization_summary.json'
previous_normalization_rows, previous_normalization_summary = load_prior_regression(prior_per_query_path, prior_summary_path)
with tempfile.TemporaryDirectory() as missing_prior_root:
    missing_prior_root = Path(missing_prior_root)
    assert load_prior_regression(missing_prior_root / 'missing.csv', missing_prior_root / 'missing.json') == (None, None)
protected_files = sorted(path for path in DATA_ROOT.glob('retrieval_*') if path.is_file() and not path.name.startswith(OUTPUT_PREFIX))
usage_before = json.loads((DATA_ROOT / 'embedding_usage.json').read_text(encoding='utf-8'))
state_before = {
    'api_requests': usage_before['executed_api_requests'], 'api_input_tokens': usage_before['api_input_tokens'],
    'embedding_cache_hash': tree_hash(DATA_ROOT / 'embedding_cache'), 'chroma_hash': tree_hash(CHROMA_ROOT), 'chroma_count': chroma_count(),
    'protected_file_hashes': {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in protected_files},
}
assert state_before['chroma_hash'] == 'f2d71eb41cfaa770516d39e670eb9ba0ef8ea489ddfb38fe869e7f3a08a1ac8b'
chunks = [json.loads(line) for line in (DATA_ROOT / 'chunks.jsonl').read_text(encoding='utf-8').splitlines()]
chunk_by_id = {chunk['id']: chunk for chunk in chunks}
baseline_rows = list(csv.DictReader((DATA_ROOT / 'retrieval_per_query.csv').open(encoding='utf-8')))
evaluation_by_id = {}
for row in baseline_rows:
    if row['method'] == 'keyword':
        evaluation_by_id[row['query_id']] = {'query_id': row['query_id'], 'query': row['query'], 'category': row['category'], 'expected_card': row['expected_card'], 'expected_level': row['expected_level'], 'required_terms': json.loads(row['required_terms'])}
assert len(evaluation_by_id) == 30
search_queries = {query_id: item['query'] for query_id, item in evaluation_by_id.items()}

def canonical_decimal(value):
    rendered = format(Decimal(str(value).replace(',', '')).normalize(), 'f')
    rendered = rendered.rstrip('0').rstrip('.') if '.' in rendered else rendered
    return '0' if rendered in {'', '-0'} else rendered


def search_tokens(value):
    text = normalized_text(value)
    tokens = list(RAW_TOKEN.findall(text))  # Preserve original normalized tokens.
    for run in re.findall(r'[가-힣](?:[가-힣 ]{0,38}[가-힣])?', text):
        joined = run.replace(' ', '')
        for size in (2, 3, 4):
            tokens.extend(f'ko{size}_{joined[index:index + size]}' for index in range(max(0, len(joined) - size + 1)))
    consumed = []
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*만\s*(\d[\d,]*(?:\.\d+)?)\s*천\s*원', text):
        amount = Decimal(match.group(1).replace(',', '')) * 10000 + Decimal(match.group(2).replace(',', '')) * 1000
        tokens.append(f'money_krw_{canonical_decimal(amount)}'); consumed.append(match.span())
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*(만|천)?\s*원', text):
        if any(left <= match.start() and match.end() <= right for left, right in consumed): continue
        multiplier = {'만': 10000, '천': 1000, None: 1}[match.group(2)]
        amount = Decimal(match.group(1).replace(',', '')) * multiplier
        tokens.append(f'money_krw_{canonical_decimal(amount)}')
    for match in re.finditer(r'(\d[\d,]*(?:\.\d+)?)\s*%', text):
        tokens.append(f'percent_{canonical_decimal(match.group(1))}')
    for match in re.finditer(r'(?:(월|연|년|일)\s*)?(\d[\d,]*(?:\.\d+)?)\s*(회|개월|년|일)', text):
        prefix = match.group(1) or 'none'; tokens.append(f'period_{prefix}_{canonical_decimal(match.group(2))}_{match.group(3)}')
    return tokens


assert canonical_decimal('1') == canonical_decimal('1.0') == canonical_decimal('1.000') == '1'
assert canonical_decimal('1,000.50') == canonical_decimal('1000.5') == '1000.5'
assert 'percent_1' in search_tokens('1%') and 'percent_1' in search_tokens('1.0%')
assert 'period_none_1_개월' in search_tokens('1개월') and 'period_none_1_개월' in search_tokens('1.0개월')
assert 'money_krw_1000.5' in search_tokens('1,000.50원') and 'money_krw_1000.5' in search_tokens('1000.5원')
assert canonical_decimal('1.5') != canonical_decimal('15')


def bm25_scores(query_tokens, documents):
    tokenized = {identifier: list(tokens) for identifier, tokens in documents.items()}
    document_frequency = Counter(token for tokens in tokenized.values() for token in set(tokens))
    average_length = sum(map(len, tokenized.values())) / len(tokenized) if tokenized else 1.0
    scores = {}
    for identifier, tokens in tokenized.items():
        frequencies, score = Counter(tokens), 0.0
        for token in query_tokens:
            frequency = frequencies[token]
            if frequency:
                inverse_frequency = math.log(1 + (len(tokenized) - document_frequency[token] + 0.5) / (document_frequency[token] + 0.5))
                score += inverse_frequency * frequency * 2.5 / (frequency + 1.5 * (0.25 + 0.75 * len(tokens) / average_length))
        scores[identifier] = score
    return scores


def rank_scores(scores):
    return [identifier for identifier, _ in sorted(scores.items(), key=lambda item: (-item[1], item[0]))]


def field_rank(query_tokens, fields, weights):
    combined = defaultdict(float)
    for field, weight in weights.items():
        for identifier, score in bm25_scores(query_tokens, fields[field]).items(): combined[identifier] += weight * score
    return rank_scores(combined)


raw_documents = {chunk['id']: RAW_TOKEN.findall(normalized_text(chunk['document'])) for chunk in chunks}
normalized_documents = {chunk['id']: search_tokens(chunk['document']) for chunk in chunks}
non_answer_fields = {
    'body': normalized_documents,
    'issuer': {chunk['id']: search_tokens(chunk['metadata'].get('issuer', '')) for chunk in chunks},
    'card_name': {chunk['id']: search_tokens(chunk['metadata'].get('card_name', '')) for chunk in chunks},
    'page_num': {chunk['id']: search_tokens(chunk['metadata'].get('page_num', '')) for chunk in chunks},
}
NON_ANSWER_WEIGHTS = {'body': 1.0, 'issuer': 1.25, 'card_name': 2.0, 'page_num': 0.25}
ORACLE_WEIGHTS = {**NON_ANSWER_WEIGHTS, 'oracle_structured': 1.5}
assert set(non_answer_fields) == set(NON_ANSWER_WEIGHTS) == {'body', 'issuer', 'card_name', 'page_num'}
assert {'structured_metadata', 'label_ids', 'label_kinds', 'section', 'level'}.isdisjoint(non_answer_fields)
oracle_fields = {**non_answer_fields, 'oracle_structured': {}}
for chunk in chunks:
    # This field contains user-verified structured label values/context and is intentionally oracle-only.
    oracle_fields['oracle_structured'][chunk['id']] = search_tokens(chunk['metadata'].get('structured_metadata', ''))

raw_bm25, normalized_bm25, metadata_bm25, oracle_bm25 = {}, {}, {}, {}
for query_id, query_text in search_queries.items():
    raw_query = RAW_TOKEN.findall(normalized_text(query_text)); canonical_query = search_tokens(query_text)
    raw_bm25[query_id] = rank_scores(bm25_scores(raw_query, raw_documents))
    normalized_bm25[query_id] = rank_scores(bm25_scores(canonical_query, normalized_documents))
    metadata_bm25[query_id] = field_rank(canonical_query, non_answer_fields, NON_ANSWER_WEIGHTS)
    oracle_bm25[query_id] = field_rank(canonical_query, oracle_fields, ORACLE_WEIGHTS)
published_raw = {row['query_id']: json.loads(row['top5_chunk_ids']) for row in baseline_rows if row['method'] == 'keyword'}
assert all(raw_bm25[query_id][:5] == published_raw[query_id] for query_id in search_queries)

published_vector = {row['query_id']: json.loads(row['top5_chunk_ids']) for row in baseline_rows if row['method'] == 'vector'}
cached_items = [(f"chunk:{chunk['id']}", chunk['document']) for chunk in chunks] + [(f"query:{query_id}", text) for query_id, text in search_queries.items()]
vectors_by_key = {}
for batch_start in range(0, len(cached_items), 64):
    batch = cached_items[batch_start:batch_start + 64]; hashes = [sha256_bytes(text.encode()) for _, text in batch]
    fingerprint = sha256_bytes(canonical_json({'model': EMBEDDING_MODEL, 'hashes': hashes}).encode())
    cached = np.load(DATA_ROOT / 'embedding_cache' / EMBEDDING_MODEL / f'{fingerprint}.npz', allow_pickle=False)
    assert cached['hashes'].tolist() == hashes
    vectors_by_key.update({key: vector for (key, _), vector in zip(batch, cached['embeddings'])})
query_vectors = {query_id: vectors_by_key[f"query:{query_id}"] for query_id in search_queries}
chroma_snapshot = tempfile.TemporaryDirectory()
snapshot_root = Path(chroma_snapshot.name) / 'chroma'
shutil.copytree(CHROMA_ROOT, snapshot_root)
chroma_client = chromadb.PersistentClient(path=str(snapshot_root), settings=Settings(anonymized_telemetry=False))
collection_name = json.loads((DATA_ROOT / 'index_manifest.json').read_text(encoding='utf-8'))['collection']
collection = chroma_client.get_collection(collection_name)
chroma_vector_rank = {}
for query_id in search_queries:
    response = collection.query(query_embeddings=[query_vectors[query_id].tolist()], n_results=min(50, len(chunks)), include=['distances'])
    chroma_vector_rank[query_id] = response['ids'][0]
assert all(chroma_vector_rank[query_id][:5] == published_vector[query_id] for query_id in search_queries)

def rrf(keyword_ranking, vector_ranking, constant=60):
    scores = defaultdict(float)
    for ranking in (keyword_ranking[:50], vector_ranking[:50]):
        for rank, identifier in enumerate(ranking, 1): scores[identifier] += 0.5 / (constant + rank)
    return rank_scores(scores)


CONFIGURATIONS = {
    'raw_text_bm25': raw_bm25, 'normalized_bm25': normalized_bm25, 'metadata_field_bm25': metadata_bm25,
    'oracle_structured_field_bm25': oracle_bm25, 'full_vector_published': published_vector,
    'rrf_vector_normalized': {query_id: rrf(normalized_bm25[query_id], chroma_vector_rank[query_id]) for query_id in search_queries},
    'rrf_vector_metadata': {query_id: rrf(metadata_bm25[query_id], chroma_vector_rank[query_id]) for query_id in search_queries},
    'rrf_vector_oracle_structured': {query_id: rrf(oracle_bm25[query_id], chroma_vector_rank[query_id]) for query_id in search_queries},
}

def relevant_ids(evaluation):
    return {chunk['id'] for chunk in chunks if chunk['metadata']['card_key'] == evaluation['expected_card'] and chunk['metadata']['level'] == evaluation['expected_level'] and all(normalized_text(term) in normalized_text(chunk['document']) for term in evaluation['required_terms'])}


def metrics(evaluation, ranking):
    relevant = relevant_ids(evaluation); hits = [identifier in relevant for identifier in ranking[:5]]
    first = next((rank for rank, hit in enumerate(hits, 1) if hit), None)
    dcg = sum(hit / math.log2(rank + 1) for rank, hit in enumerate(hits, 1)); ideal = sum(1 / math.log2(rank + 1) for rank in range(1, min(5, len(relevant)) + 1))
    return {'card_hit_at_3': int(any(chunk_by_id[item]['metadata']['card_key'] == evaluation['expected_card'] for item in ranking[:3])), 'strict_evidence_hit_at_3': int(any(hits[:3])), 'recall_at_5': sum(hits) / len(relevant), 'mrr_at_5': 1 / first if first else 0.0, 'ndcg_at_5': dcg / ideal if ideal else 0.0}


per_query_rows = []
for configuration, rankings in CONFIGURATIONS.items():
    for query_id, evaluation in evaluation_by_id.items():
        ranking = rankings[query_id]
        per_query_rows.append({'configuration': configuration, 'query_id': query_id, 'question_group': 'card' if evaluation['expected_level'] == 'card' else 'evidence', 'category': evaluation['category'], 'query': evaluation['query'], 'expected_card': evaluation['expected_card'], 'expected_level': evaluation['expected_level'], **metrics(evaluation, ranking), 'top5_chunk_ids': canonical_json(ranking[:5]), 'top5_cards': canonical_json([chunk_by_id[item]['metadata']['card_key'] for item in ranking[:5]]), 'top5_levels': canonical_json([chunk_by_id[item]['metadata']['level'] for item in ranking[:5]])})

def aggregate(rows):
    return {**{metric: sum(row[metric] for row in rows) / len(rows) for metric in ('card_hit_at_3', 'strict_evidence_hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5')}, 'denominator': len(rows)}


summary_rows = []
for configuration in CONFIGURATIONS:
    selected = [row for row in per_query_rows if row['configuration'] == configuration]
    groups = {'all': selected, 'card': [row for row in selected if row['question_group'] == 'card'], 'evidence': [row for row in selected if row['question_group'] == 'evidence'], **{f'category_{category}': [row for row in selected if row['category'] == category] for category in ('proper_noun', 'numeric_condition', 'semantic')}}
    for group, rows in groups.items(): summary_rows.append({'configuration': configuration, 'question_group': group, **aggregate(rows)})

prior_regression_loaded = previous_normalization_rows is not None
if prior_regression_loaded:
    previous_raw_rows = {row['query_id']: row for row in previous_normalization_rows if row['configuration'] == 'raw_text_bm25'}
    current_raw_rows = {row['query_id']: row for row in per_query_rows if row['configuration'] == 'raw_text_bm25'}
    assert previous_raw_rows.keys() == current_raw_rows.keys()
    for query_id in previous_raw_rows:
        for field in ('card_hit_at_3', 'strict_evidence_hit_at_3', 'recall_at_5', 'mrr_at_5', 'ndcg_at_5', 'top5_chunk_ids', 'top5_cards', 'top5_levels'):
            assert previous_raw_rows[query_id][field] == str(current_raw_rows[query_id][field])
chroma_snapshot.cleanup()
usage_after = json.loads((DATA_ROOT / 'embedding_usage.json').read_text(encoding='utf-8'))
state_after = {'api_requests': usage_after['executed_api_requests'], 'api_input_tokens': usage_after['api_input_tokens'], 'embedding_cache_hash': tree_hash(DATA_ROOT / 'embedding_cache'), 'chroma_hash': tree_hash(CHROMA_ROOT), 'chroma_count': chroma_count(), 'protected_file_hashes': {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in protected_files}}
assert state_after == state_before
result = {
    'schema_version': 'retrieval_search_normalization_v1', 'created_at': datetime.now(timezone.utc).isoformat(),
    'execution': {'network_calls': 0, 'openai_calls': 0, 'state_before': state_before, 'state_after': state_after, 'protected_assets_unchanged': True, 'raw_baseline_exact_reproduction': True, 'prior_regression_snapshot_loaded': prior_regression_loaded, 'raw_previous_output_exact': True if prior_regression_loaded else None, 'standalone_without_prior_supported': True, 'persisted_chroma_vector_top5_exact_queries': 30},
    'normalization_rules': ['NFKC + lower + whitespace normalization while preserving raw regex tokens', 'Hangul character 2/3/4-grams over runs with spaces removed', 'canonical Decimal tokens collapse representation-only trailing zeros but preserve distinct numeric values', 'KRW canonical tokens for comma/plain and 만/천 notation including decimal and 만+천 sums', 'percent canonical tokens', '월/연/년/일 prefix + 회/개월/년/일 period tokens'],
    'field_weights': {'metadata_field_bm25': NON_ANSWER_WEIGHTS, 'oracle_structured_field_bm25': ORACLE_WEIGHTS},
    'vector_source': 'Post-incident byte-for-byte temporary snapshot of current persisted local Chroma (tree hash f2d71e...) queried through collection.query with cached query embeddings, include=[distances], n_results=min(50, len(chunks)); published top-5 asserted 30/30 exact. Pre-incident ranks 6-50 were not saved, so full-ranking identity is not claimed.',
    'integrity_incident': {'source_tree_hash_before': 'a25c1d99940f74f35a220ed46c697ef49b2eb2a7e02cb66441d92e34b57e6174', 'source_tree_hash_after': 'f2d71eb41cfaa770516d39e670eb9ba0ef8ea489ddfb38fe869e7f3a08a1ac8b', 'cause': 'The first implementation connected PersistentClient directly to the protected source, allowing Chroma bookkeeping to change its byte tree.', 'verified_unchanged_semantics': {'collection_count': 327, 'collection_name': collection_name, 'published_vector_top5_exact_queries': 30}, 'unproven': 'Pre-change ranks 6-50 were not persisted, so pre/post full-vector ranking and pre-incident RRF equivalence cannot be proven.', 'current_rrf_basis': 'Post-incident f2d71e... byte snapshot.'},
    'oracle_contract': 'oracle_structured reads the existing per-benefit user-verified structured_metadata search field only. It does not modify chunk text, cache, or Chroma and is not an operational result.',
    'leakage_audit': {'status': 'passed_with_common_chunking_limitation', 'ranking_inputs_non_oracle': ['query string', 'raw chunk text', 'issuer', 'card_name', 'page_num', 'cached vectors'], 'metadata_fields_non_oracle': ['issuer', 'card_name', 'page_num'], 'forbidden_non_oracle_metadata_asserted_absent': ['structured_metadata', 'label_ids', 'label_kinds', 'section', 'level'], 'oracle_only_input': ['user-verified structured_metadata values/context'], 'evaluation_only_excluded_from_ranking': ['expected_card', 'expected_level', 'required_terms', 'category'], 'semantic_synonyms': 'none', 'implementation': 'search_queries is separated before ranking; evaluation_by_id enters only relevant_ids/metrics/grouping.', 'common_chunking_limitation': 'All configurations reuse structured-assisted chunks, so benefit boundaries and document bodies can still reflect structured boundary construction even though non-oracle metadata ranking reads only issuer/card_name/page_num.'},
    'development_set_warning': 'These 30 queries are the development set used to compare configurations, not an independent holdout.',
    'summaries': summary_rows,
    'limitations': ['Oracle structured fields contain hand-verified answers and context, so their score is an upper-bound with direct retrieval leakage.', 'All configurations share structured-assisted chunk boundaries even though non-oracle metadata fields are restricted to issuer/card_name/page_num.', 'The Chroma integrity incident left pre-change ranks 6-50 unavailable; current RRF uses the post-incident snapshot and pre-change full-ranking equality is unproven.', 'Hangul character n-grams improve spacing robustness but can increase accidental lexical matches.', 'No meaning synonym expansion or evaluation-label query expansion is used.', 'BC selected excerpt and IBK incomplete/ambiguous coverage remain.'],
}
atomic_csv(DATA_ROOT / 'retrieval_search_normalization_per_query.csv', per_query_rows)
atomic_csv(DATA_ROOT / 'retrieval_search_normalization_summary.csv', summary_rows)
atomic_json(DATA_ROOT / 'retrieval_search_normalization_summary.json', result)
print({'offline': True, 'api_calls': 0, 'raw_baseline_exact': True, 'vector_top5_exact': '30/30', 'pre_incident_top6_50': 'unproven', 'prior_regression_loaded': prior_regression_loaded, 'standalone_without_prior': True, 'metadata_fields': list(NON_ANSWER_WEIGHTS), 'vector_source': 'post-incident persisted Chroma snapshot', 'protected_unchanged': state_before == state_after, 'configurations': len(CONFIGURATIONS), 'summary_rows': len(summary_rows), 'per_query_rows': len(per_query_rows)})

{'offline': True, 'api_calls': 0, 'raw_baseline_exact': True, 'vector_top5_exact': '30/30', 'pre_incident_top6_50': 'unproven', 'prior_regression_loaded': True, 'standalone_without_prior': True, 'metadata_fields': ['body', 'issuer', 'card_name', 'page_num'], 'vector_source': 'post-incident persisted Chroma snapshot', 'protected_unchanged': True, 'configurations': 8, 'summary_rows': 48, 'per_query_rows': 240}
